# CS781 Capstone — Phase 2: Legal NER (L-NER) on 2×T4

**Kaggle settings:** Accelerator **GPU T4 ×2**, Internet **on**, secret **`HF`** (your Hugging Face token), and
**Add Input →** the dataset/notebook output that holds your Phase-1 checkpoints (`run_A/best.pt`, `run_B2/best.pt`
or `run_B/best.pt`, `run_C/best.pt`, `run_D/best.pt`) and `legal_bpe.json`.

**Your parts (course rule 9):** `heads.py`, `techniques/{lora,lorma,adapters,bitfit,partial_ft,kd}.py` (Stage 1b) and every
`None` in the configuration cell (Stage 3). Everything else is plumbing (declare it): data, char↔token alignment,
windows, training loop, strict macro-F1, prediction.json + zip.

Order: Stage 0 → 1 → 1b → 2 (data + oracle check) → 3 (configs) → 4 (runs) → 5 (results / zips).
Outputs: `/kaggle/working/phase2/runs/<name>/` (summary.json, val_scores.json, submission_<name>.zip) and all zips
collected in `/kaggle/working/submissions/`.

## Stage 0 — session, folders

In [ ]:
# session clock + folders + GPUs
import os, sys, json, glob, time, hashlib, subprocess, shutil
SESSION_START = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-crf"], check=True)   # heads.py imports torchcrf
WORK = "/kaggle/working/phase2"
for d in ("phase1_models", "techniques", "configs", "runs", "data"):
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
import torch
N_GPUS = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])

## Stage 1 — plumbing files (your Phase-1 model code unchanged; declare the rest)

In [ ]:
%%writefile /kaggle/working/phase2/a1_tokenizer.py
# Legal byte-level BPE (your A1 tokenizer, optimised)
# tokenizer.py
#
# Byte-level BPE with an end-of-word marker (A1 design), adapted for the CS781 legal corpus.
#
# Changes vs. the A1 version (original kept in Assignment3/a1_tokenizer.py):
#   * Tokens are integer ids backed by raw bytes, not strings. Previously the byte "<", "/",
#     "w", ">" could merge into a token identical to the "</w>" marker (or to "<pad>"), and
#     decode() would then insert spurious spaces. Ids make that collision impossible.
#   * Legal-text pre-tokenisation inside each whitespace word: letter runs, digit groups of
#     at most 3, and punctuation runs are merged separately ("s.302(1)," -> s . 302 ( 1 ) ,),
#     so "court", "court," and "court." share one vocabulary entry instead of three, and
#     years / section numbers / citations don't flood the vocabulary. Only the last piece of
#     each whitespace word carries </w>, so decode(encode(x)) == " ".join(x.split()).
#   * Incremental BPE training (pair counts + inverted index + lazy max-heap): each merge
#     only touches the words that contain the merged pair, instead of recounting every pair
#     in the corpus for every merge. 32k merges on a legal sample take minutes, not days.
#   * Encoding uses merge ranks (GPT-2 style, same result as applying merges in order) and a
#     bounded per-word cache (the old lru_cache on a method also cached `self` and never freed).
#   * Extra reserved special tokens for later phases (cls/sep/mask, span-corruption sentinels).
#   * encode_many() encodes many documents in parallel worker processes.
#
# Assignment 3 contract is unchanged: train_tokenizer(train_text, vocab_size) returns an object
# with encode(text) -> list[int], decode(ids) -> str and a vocab_size attribute.

import re
import os
import heapq
import string
import json
from collections import Counter, defaultdict


# ============================================================
# SPECIAL TOKENS
# ============================================================

N_SENTINELS = 25

SPECIALS = (
    ["<pad>", "<unk>", "<s>", "</s>", "<cls>", "<sep>", "<mask>"]
    + [f"<extra_id_{i}>" for i in range(N_SENTINELS)]
)   # 32 specials: ids 0..31

MIN_COUNT = 2

# Pieces inside one whitespace word: letter runs | digit groups (<= 3) | punctuation runs | underscores.
# The alternatives cover every non-space character, so "".join(pieces) == word always holds.
PRETOKENIZE = re.compile(r"[^\W\d_]+|\d{1,3}|[^\w\s]+|_+")

MAX_CACHE_WORDS = 2_000_000


# ============================================================
# TEXT CLEANING (A1 helpers, unchanged; not used by the BPE)
# ============================================================

def check_valid_en(word):
    is_en = False
    is_hi = False
    has_digit = False

    for ch in word:
        code = ord(ch)

        # English alphabet
        if 0x0041 <= code <= 0x005A or 0x0061 <= code <= 0x007A:
            is_en = True

        # Arabic numerals
        elif 0x0030 <= code <= 0x0039:
            has_digit = True

        # Devanagari numerals
        elif 0x0966 <= code <= 0x096F:
            has_digit = True

        # Devanagari
        elif 0x0900 <= code <= 0x097F:
            is_hi = True

    if has_digit:
        return False

    return is_en and not is_hi


def clean_text_and_tokenize_en(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    return [word for word in tokens if check_valid_en(word)]


# ============================================================
# BYTE-LEVEL BPE
# ============================================================

def split_word(word):
    """Whitespace word -> list of (piece, is_last_piece)."""
    pieces = PRETOKENIZE.findall(word) or [word]
    last = len(pieces) - 1
    return [(p, i == last) for i, p in enumerate(pieces)]


class ByteBPE:

    def __init__(self, vocab_size=16000):

        self.vocab_size = vocab_size

        # Id layout: specials | 256 bytes | </w> | one id per learned merge
        self.n_special = len(SPECIALS)
        self.byte_offset = self.n_special
        self.eow_id = self.byte_offset + 256

        self.merges = []            # list of (left_id, right_id); merge k creates id eow_id + 1 + k
        self._reset_tables()

    # --------------------------------------------------------
    # VOCAB TABLES
    # --------------------------------------------------------

    def _reset_tables(self):
        self.token_bytes = [b""] * self.n_special + [bytes([b]) for b in range(256)] + [b""]
        self.token_eow = [False] * (self.n_special + 256) + [True]
        for left, right in self.merges:
            self.token_bytes.append(self.token_bytes[left] + self.token_bytes[right])
            self.token_eow.append(self.token_eow[right])
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.merge_to_id = {pair: self.eow_id + 1 + rank for rank, pair in enumerate(self.merges)}
        # decode table: bytes of each token, plus a space if it ends a word; specials decode to nothing
        self._decode_table = [
            b"" if i < self.n_special else tb + (b" " if eow else b"")
            for i, (tb, eow) in enumerate(zip(self.token_bytes, self.token_eow))
        ]
        self.itos = list(SPECIALS) + [self._display(i) for i in range(self.n_special, len(self.token_bytes))]
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.vocab = self.itos
        self._cache = {}

    def _display(self, idx):
        """Human-readable token string (for inspection only; never used for encoding)."""
        text = self.token_bytes[idx].decode("utf-8", errors="backslashreplace")
        return text + ("</w>" if self.token_eow[idx] else "")

    @property
    def special_ids(self):
        return {tok: i for i, tok in enumerate(SPECIALS)}

    @property
    def eot_id(self):
        """Document separator used in the pretraining stream."""
        return SPECIALS.index("</s>")

    def get_vocab_size(self):
        return len(self.token_bytes)

    # --------------------------------------------------------
    # PIECE -> INITIAL SYMBOLS
    # --------------------------------------------------------

    def _piece_symbols(self, piece, is_last):
        symbols = [self.byte_offset + b for b in piece.encode("utf-8")]
        if is_last:
            symbols.append(self.eow_id)
        return symbols

    # --------------------------------------------------------
    # TRAIN BPE
    # --------------------------------------------------------

    @staticmethod
    def count_words(texts):
        """Whitespace-word frequencies of an iterable of texts."""
        counts = Counter()
        for text in texts:
            counts.update(str(text).split())
        return counts

    def fit(self, tokenized_corpus=None, target_vocab_size=None, min_count=MIN_COUNT,
            word_counts=None, min_word_freq=1, verbose=True):
        """Learn merges.

        tokenized_corpus: iterable of lists of whitespace words (A1/A3 interface), or
        word_counts:      a precomputed {whitespace word: frequency} mapping (faster for big corpora).
        min_word_freq:    ignore whitespace words rarer than this while learning merges (speed knob).
        """
        target = self.vocab_size if target_vocab_size is None else target_vocab_size
        self.vocab_size = target

        if word_counts is None:
            word_counts = Counter()
            for sentence_tokens in tokenized_corpus:
                word_counts.update(sentence_tokens)

        # whitespace words -> (piece, is_last) frequencies
        piece_freq = Counter()
        for word, count in word_counts.items():
            if count < min_word_freq:
                continue
            for piece in split_word(word):
                piece_freq[piece] += count

        words = [self._piece_symbols(p, last) for p, last in piece_freq]
        freqs = list(piece_freq.values())
        del piece_freq

        if verbose:
            print("=" * 60)
            print("BUILDING BYTE-LEVEL BPE")
            print("=" * 60)
            print(f"Unique pre-token pieces: {len(words):,}")
            print(f"Initial vocabulary size: {self.eow_id + 1:,}")
            print(f"Target vocabulary size: {target:,}")

        # pair -> total count, pair -> set of word indices containing it
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for wi, (symbols, f) in enumerate(zip(words, freqs)):
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += f
                where[pair].add(wi)

        # max-heap on count with deterministic tie-break on ids; stale entries are skipped lazily
        heap = [(-c, pair) for pair, c in pair_counts.items()]
        heapq.heapify(heap)

        self.merges = []
        next_id = self.eow_id + 1

        while next_id < target and heap:
            neg, pair = heapq.heappop(heap)
            current = pair_counts.get(pair, 0)
            if current != -neg:
                if current > 0:
                    heapq.heappush(heap, (-current, pair))
                continue
            if current < min_count:
                if verbose:
                    print(f"\nStopping: best pair frequency = {current}")
                break

            left, right = pair
            new_id = next_id
            next_id += 1
            self.merges.append(pair)
            touched = Counter()

            for wi in list(where.pop(pair, ())):
                symbols, f = words[wi], freqs[wi]
                # remove this word's old pairs
                for p in zip(symbols, symbols[1:]):
                    pair_counts[p] -= f
                    touched[p] += 0
                    s = where.get(p)
                    if s is not None:
                        s.discard(wi)
                # merge left-to-right
                merged, i, n = [], 0, len(symbols)
                while i < n:
                    if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                        merged.append(new_id)
                        i += 2
                    else:
                        merged.append(symbols[i])
                        i += 1
                words[wi] = merged
                # add the new pairs
                for p in zip(merged, merged[1:]):
                    pair_counts[p] += f
                    touched[p] += 0
                    where[p].add(wi)

            pair_counts.pop(pair, None)
            for p in touched:
                c = pair_counts.get(p, 0)
                if c <= 0:
                    pair_counts.pop(p, None)
                    where.pop(p, None)
                elif new_id in p:
                    heapq.heappush(heap, (-c, p))   # only pairs with the new symbol can grow

            if verbose and len(self.merges) % 1000 == 0:
                print(f"Merges: {len(self.merges):,} | Vocab: {next_id:,} | Latest pair count: {current:,}")

        self._reset_tables()

        if verbose:
            print("\n" + "=" * 60)
            print("BPE TRAINING COMPLETE")
            print("=" * 60)
            print(f"Final vocabulary size: {self.get_vocab_size():,}")
            print(f"Total merges learned: {len(self.merges):,}")

        return self

    # --------------------------------------------------------
    # ENCODE
    # --------------------------------------------------------

    def _bpe(self, symbols):
        """Repeatedly merge the lowest-rank adjacent pair (== applying merges in learned order)."""
        ranks = self.ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for pair in zip(symbols, symbols[1:]):
                r = ranks.get(pair)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = pair, r
            if best is None:
                break
            new_id = self.merge_to_id[best]
            left, right = best
            merged, i, n = [], 0, len(symbols)
            while i < n:
                if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(symbols[i])
                    i += 1
            symbols = merged
        return symbols

    def encode_word(self, word):
        ids = self._cache.get(word)
        if ids is None:
            ids = []
            for piece, is_last in split_word(word):
                ids.extend(self._bpe(self._piece_symbols(piece, is_last)))
            ids = tuple(ids)
            if len(self._cache) >= MAX_CACHE_WORDS:
                self._cache.clear()
            self._cache[word] = ids
        return ids

    def encode_tokens(self, sentence_tokens):
        ids = []
        for word in sentence_tokens:
            ids.extend(self.encode_word(word))
        return ids

    def encode(self, text):
        """text -> list of integer ids (whitespace is normalised to single spaces)."""
        cache = self._cache
        ids = []
        for word in str(text).split():
            w = cache.get(word)
            if w is None:
                w = self.encode_word(word)
            ids.extend(w)
        return ids

    def pool(self, workers=None):
        """Process pool whose workers each hold a copy of this tokenizer (and keep their word cache).
        Use with the module-level worker functions, e.g. pool.imap(encode_array_worker, texts)."""
        import multiprocessing as mp
        workers = workers or os.cpu_count() or 1
        return mp.get_context("fork").Pool(workers, initializer=_worker_init, initargs=(self,))

    def encode_many(self, texts, workers=None, chunksize=64):
        """Encode many documents in parallel processes; yields lists of ids in input order."""
        with self.pool(workers) as pool:
            yield from pool.imap(_worker_encode, texts, chunksize=chunksize)

    # --------------------------------------------------------
    # DECODE
    # --------------------------------------------------------

    def decode(self, ids):
        table = self._decode_table
        n = len(table)
        out = b"".join(table[int(i)] if 0 <= int(i) < n else b"" for i in ids)
        return out.decode("utf-8", errors="replace").strip()

    def ends_word(self, idx):
        """True if token idx closes a whitespace word (used to count generated words)."""
        return self.token_eow[idx]

    # --------------------------------------------------------
    # SAVE / LOAD
    # --------------------------------------------------------

    def to_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "format": "legal-bytebpe-v2",
                    "specials": SPECIALS,
                    "merges": self.merges,
                    "vocab_size": self.get_vocab_size(),
                    "itos": self.itos,          # readable, informational only
                },
                f,
                ensure_ascii=False,
            )
        print(f"Saved tokenizer to: {path}")

    @classmethod
    def from_json(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if data.get("format") != "legal-bytebpe-v2":
            raise ValueError(f"{path} was not written by this tokenizer version")
        if data["specials"] != SPECIALS:
            raise ValueError("special tokens differ from this tokenizer version")
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = [tuple(pair) for pair in data["merges"]]
        tokenizer._reset_tables()
        return tokenizer


# worker-process helpers for encode_many (module level so they pickle)
_WORKER_TOKENIZER = None


def _worker_init(tokenizer):
    global _WORKER_TOKENIZER
    _WORKER_TOKENIZER = tokenizer


def _worker_encode(text):
    return _WORKER_TOKENIZER.encode(text)


def encode_array_worker(text):
    """(uint16 id array, whitespace word count) for one document; for tokenizer.pool().imap."""
    import numpy as np
    return np.asarray(_WORKER_TOKENIZER.encode(text), dtype=np.uint16), text.count(" ") + 1


# ============================================================
# ASSIGNMENT 3 TRAINING FUNCTION
# ============================================================

def train_tokenizer(train_text: str, vocab_size: int):
    text = str(train_text)
    print(f"Training tokenizer on {len(text):,} characters")
    tokenizer = ByteBPE(vocab_size=vocab_size)
    tokenizer.fit(
        word_counts=ByteBPE.count_words(text.splitlines()),
        target_vocab_size=vocab_size,
        min_count=MIN_COUNT,
    )
    return tokenizer


# ============================================================
# SIMPLE SMOKE TEST
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("TOKENIZER SMOKE TEST")
    print("=" * 60)

    train_text = """
    the court has jurisdiction over this matter under Section 302(1) of the Code.
    the court shall consider the evidence, vide AIR 1966 SC 1882, dated 18.12.1961.
    the evidence presented before the court </w> <pad> was 12,345 pages.
    """

    tokenizer = train_tokenizer(train_text, vocab_size=400)

    for test_text in [
        "the court has jurisdiction",
        "Section 302(1), AIR 1966 SC 1882 </w><pad> naïve — ₹5,00,000/- (u/s 138)",
    ]:
        ids = tokenizer.encode(test_text)
        decoded = tokenizer.decode(ids)
        print("\nInput:  ", test_text)
        print("Ids:    ", ids)
        print("Pieces: ", [tokenizer.itos[i] for i in ids])
        print("Decoded:", decoded)
        assert decoded == " ".join(test_text.split()), "round trip failed"

    print("\nVocabulary size:", tokenizer.get_vocab_size())

    tokenizer.to_json("bpe_smoke_test.json")
    loaded = ByteBPE.from_json("bpe_smoke_test.json")
    assert loaded.encode(test_text) == tokenizer.encode(test_text)
    os.remove("bpe_smoke_test.json")

    print("\n" + "=" * 60)
    print("SMOKE TEST PASSED")
    print("=" * 60)

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/rope.py
# your rope.py (two lines fixed, marked FIX (Claude))
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/arch1.py
# Architecture 1 — attention1.py + decoder1.py (your code)
# ======== attention1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Multi-head causal self-attention, implemented from raw tensor ops."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # Hook for Q1.4. None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        # TODO: define self.q_proj, self.k_proj, self.v_proj, self.out_proj as nn.Linear
        # (no bias). All should map d_model -> d_model.
        self.q_proj = nn.Linear(d_model,d_model,bias=False)
        self.k_proj = nn.Linear(d_model,d_model,bias=False)
        self.v_proj = nn.Linear(d_model,d_model,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # TODO: register a (max_seq_len, max_seq_len) lower-triangular boolean mask as a
        # buffer named `causal_mask` (True = allowed to attend, False = masked out), so
        # you don't rebuild it on every forward call.
        causal_mask = torch.tril(
            torch.ones(max_seq_len,max_seq_len,dtype=torch.bool)
        )
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        # TODO 1: project to q, k, v  (each (B, T, C))
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        # TODO 2: reshape each to (B, n_heads, T, d_head) — split C into (n_heads, d_head)
        #         and move n_heads next to the batch dim.
        q = torch.reshape(q,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        k = torch.reshape(k,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        v = torch.reshape(v,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        
        # TODO 3: compute attention scores = (q @ k^T) * self.attention_scale
        #         -> (B, n_heads, T, T). Do NOT hard-code 1/sqrt(d_head): Q1.4
        #         changes this hook for the scaling ablation.
        attention_scores = (q@k.transpose(2,3))*self.attention_scale
        # TODO 4: apply the causal mask (set disallowed positions to -inf before softmax)
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
        # TODO 5: softmax over the last dim, apply self.attn_dropout
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
        # TODO 6: weighted sum with v -> (B, n_heads, T, d_head)
        weighted_sum = attention_weights@v
        # TODO 7: reshape back to (B, T, C) and apply self.out_proj + self.resid_dropout
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM, built from scratch (no nn.Transformer*)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          # == n_heads for plain MHA; used in Question 2
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # or "rope", used in Question 3
    attention_scale: float = None      # Q1.4 hook; None -> 1/sqrt(d_head)
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        # TODO: implement pre-norm when norm_first=True,
        # cfg.norm_first = true for pre norm
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # (RoPE is applied inside attention instead of here — see Question 3)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/arch3.py
# Architecture 2 option — rope.py + attention3.py + decoder3.py (your code: GQA + RoPE)
# ======== rope.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated


# ======== attention3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/attention3.py
# your attention3.py (verbatim)
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/non_causal_self_attention.py
# your non_causal_self_attention.py (verbatim)
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/cross_attention.py
# your cross_attention.py (verbatim)
"""Cross Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x , memory):
        B, T, C = x.shape
        _,S,_ = memory.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/encoder.py
# your encoder.py (verbatim)
"""Encoder-Block of  Transformer LM -- Block now passes use_rope through to attention, and
EncoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from non_causal_self_attention import SelfAttention


@dataclass
class EncLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class EncBlock(nn.Module):
    def __init__(self, cfg: EncLMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = SelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class EncoderLM(nn.Module):
    def __init__(self, cfg: EncLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        #self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        #self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        #x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/decoder4.py
# your decoder4.py (verbatim)
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention3 import CausalSelfAttention
from cross_attention import CrossAttention


@dataclass
class DecLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class DecBlock(nn.Module):
    def __init__(self, cfg: DecLMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.crattn = CrossAttention(cfg.d_model , cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,
        cfg.dropout,use_rope=False,attention_scale=cfg.attention_scale)

        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x , memory):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.crattn(y3,memory)+y2
            y5 = self.ln3(y4)
            y6 = self.ffn(y5)+y4
            output=y6
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.crattn(y2,memory)
            y4 = self.ln2(y3+y2)
            y5 = self.ffn(y4)
            y6 = self.ln3(y5+y4)
            output=y6
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: DecLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([DecBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor , memory:torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x,memory)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/enc_dec.py
# your enc_dec.py (verbatim)
from dataclasses import dataclass 
import torch  
import torch.nn as nn  
from encoder import EncBlock, sinusoidal_positional_encoding  
from decoder4 import DecBlock

@dataclass
class EncLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


@dataclass
class DecLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


class EncDecLM(nn.Module):
    def __init__(self,cfg1:EncLMConfig,cfg2:DecLMConfig):
        super().__init__()
        # Encoder Block Initializations 
        self.cfg1 = cfg1
        self.tok_emb1 = nn.Embedding(cfg1.vocab_size, cfg1.d_model)
        if cfg1.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg1.max_seq_len, cfg1.d_model)
            self.register_buffer("pos_emb_enc", pe, persistent=False)
   
        self.encblocks = nn.ModuleList([EncBlock(cfg1) for _ in range(cfg1.n_layers)])
        self.ln_f_enc = nn.LayerNorm(cfg1.d_model)
        # Decoder Block Initializations
        self.cfg2 = cfg2
        self.tok_emb2 = nn.Embedding(cfg2.vocab_size, cfg2.d_model)
        if cfg2.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg2.max_seq_len, cfg2.d_model)
            self.register_buffer("pos_emb_dec", pe, persistent=False)
       
        self.decblocks = nn.ModuleList([DecBlock(cfg2) for _ in range(cfg2.n_layers)])
        self.ln_f_dec = nn.LayerNorm(cfg2.d_model)
        self.head = nn.Linear(cfg2.d_model, cfg2.vocab_size, bias=False)
        self.head.weight = self.tok_emb2.weight
    def encode(self,src:torch.Tensor)->torch.Tensor:
        # passing it through encoder 
        B,S = src.shape
        
        x_enc = self.tok_emb1(src)
        if(self.cfg1.pos_emb_type == "sinusoidal"):
            x_enc=x_enc+self.pos_emb_enc[:S]
        for block in self.encblocks:
            x_enc = block(x_enc)
        x_enc = self.ln_f_enc(x_enc)
        memory = x_enc
        return memory

    def decode(self,tgt:torch.Tensor,memory:torch.Tensor)->torch.Tensor:
        B,T = tgt.shape 
        x_dec = self.tok_emb2(tgt)
        if(self.cfg2.pos_emb_type == "sinusoidal"):
            x_dec=x_dec+self.pos_emb_dec[:T]
        for block in self.decblocks:
            x_dec = block(x_dec,memory)
        x_dec = self.ln_f_dec(x_dec)
        x_dec = self.head(x_dec)
        return x_dec


    def forward(self ,src:torch.Tensor , tgt:torch.Tensor)->torch.Tensor:
        return self.decode(tgt,self.encode(src))
        
         
        






        
    
        

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/prefix_attention.py
# your prefix_attention.py (verbatim)
""" Prefix Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
     

    def forward(self, x , p):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale

        mask1 = self.causal_mask[:T,:T].clone()
        mask1[:p,:p]=1

        attention_scores = attention_scores.masked_fill(
            ~mask1,float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/prefix_lm.py
# your prefix_lm.py (verbatim)
"""Prefix LM - Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2). Different Architecture as mask changes the way information flows"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from prefix_attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False
    prefix:int = 128


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x,p):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1,p)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x,p)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x,self.cfg.prefix)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/fastattn_ab.py
# Plumbing: fused SDPA attention + KV cache + generation around your modules
"""Efficiency plumbing around the hand-written A3 models (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay
exactly as defined in arch1/arch2/arch3. `patch_model` swaps each attention module's forward
for one that computes the *same* function with fused kernels, and adds an optional KV cache
for generation. `check_equivalence` in the notebook verifies outputs match the original code.

* Training: F.scaled_dot_product_attention(is_causal=True). On T4 (sm_75) PyTorch cannot use
  the FlashAttention-2 kernel (needs sm_80+); it dispatches to the memory-efficient fused
  kernel, which gives the same benefit: no (T x T) score matrix materialised in HBM.
* Generation: per-layer KV cache (keys/values for MHA/GQA, the compressed latent for MLA),
  prefill once, then one token per step.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forward (drop-in for all three hand-written attention classes)
# ---------------------------------------------------------------------------

def _sdpa(q, k, v, causal, dropout_p, scale):
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=scale)


def _fast_attention_forward(self, x):
    B, T, C = x.shape
    cache = getattr(self, "_kv_cache", None)
    start = cache["pos"] if cache is not None else 0
    n_heads, d_head = self.n_heads, self.d_head
    dropout_p = self.attn_dropout.p if self.training else 0.0

    q = self.q_proj(x).view(B, T, n_heads, d_head).transpose(1, 2)

    if hasattr(self, "kv_down_proj"):                       # MLA: cache the shared latent
        latent = self.kv_down_proj(x)
        if cache is not None:
            cache["latent"][:, start:start + T] = latent
            latent = cache["latent"][:, :start + T]
        S = latent.shape[1]
        k = self.k_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
        v = self.v_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
    else:                                                   # MHA / GQA (+ optional RoPE)
        n_kv = getattr(self, "n_kv_heads", n_heads)
        k = self.k_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        if getattr(self, "use_rope", False):
            cos = self.rope_cos[start:start + T]
            sin = self.rope_sin[start:start + T]
            q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
            k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
        if cache is not None:
            cache["k"][:, :, start:start + T] = k
            cache["v"][:, :, start:start + T] = v
            k = cache["k"][:, :, :start + T]
            v = cache["v"][:, :, :start + T]
        n_rep = n_heads // n_kv
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

    # prefill (start == 0) is causal; a single decode step attends to the whole cache
    causal = T > 1
    if causal and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    y = _sdpa(q, k, v, causal, dropout_p, self.attention_scale)
    y = y.transpose(1, 2).reshape(B, T, n_heads * d_head)
    return self.resid_dropout(self.out_proj(y))


def attention_modules(model):
    return [block.attn for block in model.blocks]


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    for attn in attention_modules(model):
        attn._kv_cache = None
        if getattr(attn, "use_rope", False):
            # the user's own apply_rope, from the module that defines this attention class
            attn._apply_rope = sys.modules[type(attn).__module__].apply_rope
        attn.forward = types.MethodType(_fast_attention_forward, attn)
    return model


# ---------------------------------------------------------------------------
# KV cache
# ---------------------------------------------------------------------------

@contextmanager
def kv_cache(model, batch_size, max_len, dtype):
    """Allocate a per-layer cache. Inside the context call `step(model, ids)` repeatedly."""
    dev = next(model.parameters()).device
    caches = []
    for attn in attention_modules(model):
        c = {"pos": 0}
        if hasattr(attn, "kv_down_proj"):
            c["latent"] = torch.empty(batch_size, max_len, attn.kv_latent_dim, device=dev, dtype=dtype)
        else:
            n_kv = getattr(attn, "n_kv_heads", attn.n_heads)
            shape = (batch_size, n_kv, max_len, attn.d_head)
            c["k"] = torch.empty(shape, device=dev, dtype=dtype)
            c["v"] = torch.empty(shape, device=dev, dtype=dtype)
        attn._kv_cache = c
        caches.append(c)
    full_pe = getattr(model, "pos_emb", None)
    model._fast_state = {"caches": caches, "full_pe": full_pe}
    try:
        yield
    finally:
        for attn in attention_modules(model):
            attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe
        del model._fast_state


def step(model, ids):
    """Run `ids` (B, T) through the model at the current cache position; returns logits (B, T, V)."""
    state = model._fast_state
    pos = state["caches"][0]["pos"]
    if state["full_pe"] is not None:
        # DecoderLM.forward adds self.pos_emb[:T]; shift the table so rows pos..pos+T are used
        model.pos_emb = state["full_pe"][pos:]
    logits = model(ids)
    for c in state["caches"]:
        c["pos"] += ids.shape[1]
    return logits


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p          # keep the smallest set with mass >= top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def ngram_ban(seq, n, vocab):
    """(B, V) bool: tokens that would complete an n-gram (in token ids) already present in `seq` (B, S).
    For each row, every earlier occurrence of the last n-1 tokens bans the token that followed it."""
    B, S = seq.shape
    if n < 2 or S < n:
        return torch.zeros(B, vocab, dtype=torch.bool, device=seq.device)
    grams = seq.unfold(1, n, 1)                                   # (B, S-n+1, n)
    hit = (grams[:, :, :-1] == seq[:, None, S - n + 1:]).all(-1)  # earlier (n-1)-gram == current suffix
    cnt = torch.zeros(B, vocab, device=seq.device).scatter_add_(1, grams[:, :, -1], hit.float())
    return cnt > 0


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens,
                   batch_size, temperature, top_k, top_p, repetition_penalty, device, amp_dtype,
                   log_every=10, no_repeat_ngram=0, ngram_scope="output"):
    """no_repeat_ngram = n > 1 forbids repeating any n-gram of TOKEN ids (HF no_repeat_ngram_size), counted over
    the generated text only (ngram_scope="output") or over prompt + generated text ("all")."""
    assert ngram_scope in ("output", "all"), ngram_scope
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    Prompts are bucketed by token length so a batch needs no padding: every prompt in a batch
    keeps its last L tokens, L = shortest prompt in the batch (capped so prompt + new tokens fit
    max_seq_len). Words are counted from the tokenizer's end-of-word flag, so we stop as soon as
    every row has produced target_words complete words.
    """
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    assert max_prompt > 0, "max_new_tokens must be smaller than max_seq_len"
    results = [None] * len(prompts)

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, B, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            history = ids
            for t in range(max_new_tokens):
                # a bare </w> right after a finished word never occurs in encoded text: forbid it
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = history if ngram_scope == "all" else (torch.stack(out, 1) if out else ids[:, :0])
                    ban = ngram_ban(seq, no_repeat_ngram, logits.size(1))
                    ban &= ~(ban | torch.isinf(logits)).all(-1, keepdim=True)  # never leave a row with no option
                    logits = logits.masked_fill(ban, -float("inf"))
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                # a word ends on an end-of-word token that closes something non-empty
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()):
                    break
                if repetition_penalty != 1.0 or (no_repeat_ngram and ngram_scope == "all"):
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(model, nxt[:, None])[:, -1]
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


@torch.inference_mode()
def generate_beam(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens, batch_size,
                  beam_size, device, amp_dtype, no_repeat_ngram=0, ngram_scope="output", length_penalty=1.0,
                  log_every=10):
    """Beam search with the same word budget / token constraints as generate_words.
    Beams are ranked by sum(log p) / length**length_penalty (1.0 = mean log-prob per token). A beam that has
    produced `target_words` words is frozen (score and length fixed) and keeps competing as a finished
    hypothesis. Optional n-gram blocking per beam (same rule as generate_words). Returns list[str] in input order."""
    assert ngram_scope in ("output", "all"), ngram_scope
    model.eval()
    W = int(beam_size)
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    n_decodable = len(tokenizer.token_bytes)
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    per_batch = max(1, batch_size // W)
    results = [None] * len(prompts)
    neg_inf = -float("inf")

    for b, start in enumerate(range(0, len(order), per_batch)):
        idxs = order[start:start + per_batch]
        P = len(idxs)
        R = P * W
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        prompt_ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        ids = prompt_ids.repeat_interleave(W, 0)                                   # (R, L): row = p * W + beam
        words = torch.zeros(R, dtype=torch.int32, device=device)
        in_word = torch.zeros(R, dtype=torch.bool, device=device)
        done = torch.zeros(R, dtype=torch.bool, device=device)
        cum = torch.zeros(R, device=device)
        length = torch.zeros(R, device=device)
        gen = torch.zeros(R, 0, dtype=torch.long, device=device)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, R, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            for t in range(max_new_tokens):
                lp = torch.nan_to_num(logits.float(), nan=-1e4)
                lp[:, :tokenizer.n_special] = neg_inf
                lp[~in_word, tokenizer.eow_id] = neg_inf
                lp[:, n_decodable:] = neg_inf
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = gen if ngram_scope == "output" else torch.cat([ids, gen], 1)
                    ban = ngram_ban(seq, no_repeat_ngram, lp.size(1))
                    ban &= ~(ban | torch.isinf(lp)).all(-1, keepdim=True)
                    lp = lp.masked_fill(ban, neg_inf)
                lp = lp.log_softmax(-1)
                V = lp.size(1)
                # candidate scores (length-normalised); finished beams keep their score via one frozen continuation
                cand = (cum[:, None] + lp) / (length[:, None] + 1).pow(length_penalty)
                frozen_tok = lp.argmax(-1)
                frozen = torch.full_like(cand, neg_inf)
                frozen.scatter_(1, frozen_tok[:, None], (cum / length.clamp(min=1).pow(length_penalty))[:, None])
                cand = torch.where(done[:, None], frozen, cand)
                if t == 0:                                                         # identical beams: expand beam 0 only
                    cand.view(P, W, V)[:, 1:] = neg_inf
                top_s, top_i = cand.view(P, W * V).topk(W, dim=-1)                 # (P, W)
                parent = (top_i // V) + torch.arange(P, device=device)[:, None] * W
                parent, tok = parent.reshape(-1), (top_i % V).reshape(-1)
                was_done = done[parent]
                # reorder beam state + KV cache to the chosen parents
                for c in model._fast_state["caches"]:
                    pos = c["pos"]
                    c["k"][:, :, :pos] = c["k"][parent, :, :pos]
                    c["v"][:, :, :pos] = c["v"][parent, :, :pos]
                words, in_word, done = words[parent], in_word[parent], done[parent].clone()
                cum = torch.where(was_done, cum[parent], cum[parent] + lp[parent, tok])
                length = torch.where(was_done, length[parent], length[parent] + 1)
                gen = torch.cat([gen[parent], tok[:, None]], 1)
                open_ = in_word | has_bytes[tok]
                words = words + (eow[tok] & open_ & ~was_done).int()
                in_word = open_ & ~eow[tok]
                done = was_done | (words >= target_words)
                if bool(done.all()):
                    break
                logits = step(model, tok[:, None])[:, -1]
        score = (cum / length.clamp(min=1).pow(length_penalty)).view(P, W)
        score = torch.where(done.view(P, W), score, score - 1e6)                     # prefer finished beams
        best = score.argmax(-1) + torch.arange(P, device=device) * W
        rows = gen[best].tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(rows[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  beam generated {min(start + per_batch, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence(make_model, vocab_size, seq_len=48, rtol=1e-4):
    """Original forward vs patched forward vs KV-cached incremental decoding, fp32 on CPU.
    Returns max |difference| relative to the largest |logit| of the original model."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    with kv_cache(fast, 2, seq_len, torch.float32):
        pre = step(fast, idx[:, :seq_len // 2])
        steps = [step(fast, idx[:, t:t + 1]) for t in range(seq_len // 2, seq_len)]
    err_cache = (a - torch.cat([pre] + steps, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile /kaggle/working/phase2/phase1_models/fastattn_cd.py
# Plumbing: fused SDPA attention + KV caches + generation around your #3 / #4 modules
"""Efficiency plumbing around the hand-written encoder-decoder (#3) and prefix-LM (#4) (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay exactly as defined in
your files. `patch_model` swaps each attention module's forward for one that computes the *same* function with
F.scaled_dot_product_attention (memory-efficient fused kernel on T4), and adds optional caches for generation:

  encoder SelfAttention (non_causal_self_attention.py)  -> bidirectional, no cache (the encoder runs once per prompt)
  decoder CausalSelfAttention (attention3.py)           -> causal, self-attention KV cache
  CrossAttention (cross_attention.py)                   -> K/V of the encoder memory computed once per prompt, cached
  prefix CausalSelfAttention (prefix_attention.py)      -> mask = causal OR [:p, :p] block, KV cache for decoding

`check_equivalence_*` (run in the notebook before training) verify the patched model against your original forward,
and cached step-by-step decoding against a full forward pass.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forwards
# ---------------------------------------------------------------------------

def _dropout_p(self):
    return self.attn_dropout.p if self.training else 0.0


def _project_self(self, x, start):
    """q, k, v of a self-attention module, RoPE at absolute positions start..start+T (as in your code)."""
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
    if getattr(self, "use_rope", False):
        cos = self.rope_cos[start:start + T]
        sin = self.rope_sin[start:start + T]
        q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
        k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
    return q, k, v


def _use_cache(self, k, v, start):
    cache = getattr(self, "_kv_cache", None)
    if cache is None:
        return k, v
    T = k.shape[2]
    cache["k"][:, :, start:start + T] = k
    cache["v"][:, :, start:start + T] = v
    return cache["k"][:, :, :start + T], cache["v"][:, :, :start + T]


def _expand(self, k, v):
    if self.n_rep > 1:
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
    return k, v


def _finish(self, y):
    B, H, T, D = y.shape
    return self.resid_dropout(self.out_proj(y.transpose(1, 2).reshape(B, T, H * D)))


def _sdpa(self, q, k, v, attn_mask=None, is_causal=False):
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal,
                                          dropout_p=_dropout_p(self), scale=self.attention_scale)


def _start(self):
    cache = getattr(self, "_kv_cache", None)
    return cache["pos"] if cache is not None else 0


def _fwd_bidirectional(self, x):
    """encoder SelfAttention: no mask."""
    q, k, v = _project_self(self, x, 0)
    k, v = _expand(self, k, v)
    return _finish(self, _sdpa(self, q, k, v))


def _fwd_causal(self, x):
    """decoder CausalSelfAttention: causal; one-token steps attend to the whole cache."""
    start = _start(self)
    q, k, v = _project_self(self, x, start)
    k, v = _use_cache(self, k, v, start)
    k, v = _expand(self, k, v)
    T = x.shape[1]
    if T > 1 and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    return _finish(self, _sdpa(self, q, k, v, is_causal=T > 1))


def _prefix_mask(self, T, p, device):
    key = (T, int(p), str(device))
    masks = self.__dict__.setdefault("_mask_cache", {})
    if key not in masks:
        m = torch.ones(T, T, dtype=torch.bool, device=device).tril()
        m[:p, :p] = True                       # same rule as your prefix_attention.py
        masks[key] = m
    return masks[key]


def _fwd_prefix(self, x, p):
    """prefix attention: causal OR [:p, :p]; one-token decoding steps attend to the whole cache."""
    start = _start(self)
    q, k, v = _project_self(self, x, start)
    k, v = _use_cache(self, k, v, start)
    k, v = _expand(self, k, v)
    T = x.shape[1]
    if start == 0:
        return _finish(self, _sdpa(self, q, k, v, attn_mask=_prefix_mask(self, T, p, x.device)))
    if T == 1:
        return _finish(self, _sdpa(self, q, k, v))
    raise ValueError("multi-token steps are only supported from an empty cache")


def _fwd_cross(self, x, memory):
    """CrossAttention: q from the decoder state, k/v from the encoder memory (cached once per prompt)."""
    assert not getattr(self, "use_rope", False), "RoPE in cross-attention is not supported (your design: use_rope=False)"
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
    cache = getattr(self, "_kv_cache", None)
    if cache is not None and "k" in cache:
        k, v = cache["k"], cache["v"]
    else:
        S = memory.shape[1]
        k = self.k_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        if cache is not None:
            cache["k"], cache["v"] = k, v
    k, v = _expand(self, k, v)
    return _finish(self, _sdpa(self, q, k, v))


_FORWARDS = {
    ("non_causal_self_attention", "SelfAttention"): _fwd_bidirectional,
    ("attention3", "CausalSelfAttention"): _fwd_causal,
    ("cross_attention", "CrossAttention"): _fwd_cross,
    ("prefix_attention", "CausalSelfAttention"): _fwd_prefix,
}


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    n = 0
    for mod in model.modules():
        fwd = _FORWARDS.get((type(mod).__module__, type(mod).__name__))
        if fwd is None:
            continue
        mod._kv_cache = None
        if getattr(mod, "use_rope", False):
            mod._apply_rope = sys.modules[type(mod).__module__].apply_rope   # your own apply_rope
        mod.forward = types.MethodType(fwd, mod)
        n += 1
    assert n > 0, "no attention module recognised; did a module/class name change?"
    return model


# ---------------------------------------------------------------------------
# generation sessions: prefill once, then one token per step
# ---------------------------------------------------------------------------

def _alloc_self_cache(attn, B, max_len, dtype, device):
    shape = (B, attn.n_kv_heads, max_len, attn.d_head)
    attn._kv_cache = {"pos": 0, "k": torch.empty(shape, device=device, dtype=dtype),
                      "v": torch.empty(shape, device=device, dtype=dtype)}
    return attn._kv_cache


@contextmanager
def prefix_session(model, ids, max_new_tokens, dtype):
    """#4: prefill the prompt (mask uses the model's own cfg.prefix), then decode with the KV cache."""
    B, L = ids.shape
    dev = ids.device
    caches = [_alloc_self_cache(b.attn, B, L + max_new_tokens, dtype, dev) for b in model.blocks]
    full_pe = getattr(model, "pos_emb", None)

    def run(tok):
        pos = caches[0]["pos"]
        if full_pe is not None:
            model.pos_emb = full_pe[pos:]     # DecoderLM adds pos_emb[:T]; shift it to rows pos..pos+T
        logits = model(tok)
        for c in caches:
            c["pos"] += tok.shape[1]
        return logits[:, -1]

    try:
        yield run(ids), run
    finally:
        for b in model.blocks:
            b.attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe


@contextmanager
def encdec_session(model, ids, max_new_tokens, dtype, start_id=None):
    """#3: encode the prompt once, cache every decoder layer's cross-attention K/V, decode token by token.
    start_id=None -> the decoder starts from the last source token, else from start_id (e.g. <s>)."""
    B = ids.shape[0]
    dev = ids.device
    memory = model.encode(ids)
    caches = [_alloc_self_cache(b.attn, B, max_new_tokens + 1, dtype, dev) for b in model.decblocks]
    for b in model.decblocks:
        b.crattn._kv_cache = {}
    full_pe = getattr(model, "pos_emb_dec", None)

    def run(tok):
        pos = caches[0]["pos"]
        if full_pe is not None:
            model.pos_emb_dec = full_pe[pos:]
        logits = model.decode(tok, memory)
        for c in caches:
            c["pos"] += tok.shape[1]
        return logits[:, -1]

    first = ids[:, -1:] if start_id is None else torch.full((B, 1), start_id, device=dev, dtype=ids.dtype)
    try:
        yield run(first), run
    finally:
        for b in model.decblocks:
            b.attn._kv_cache = None
            b.crattn._kv_cache = None
        if full_pe is not None:
            model.pos_emb_dec = full_pe


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, kind, max_prompt, target_words, max_new_tokens, batch_size,
                   temperature, top_k, top_p, repetition_penalty, device, amp_dtype, start_id=None, log_every=10):
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    kind = "prefix" (#4) or "encdec" (#3). Prompts are bucketed by token length so a batch needs no padding: each
    prompt keeps its last L tokens, L = shortest prompt in the batch (capped at max_prompt)."""
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    results = [None] * len(prompts)
    cache_dtype = amp_dtype if device.type == "cuda" else torch.float32

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        session = (prefix_session(model, ids, max_new_tokens, cache_dtype) if kind == "prefix"
                   else encdec_session(model, ids, max_new_tokens, cache_dtype, start_id))
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), session as (logits, step):
            history = ids
            for t in range(max_new_tokens):
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")       # a bare </w> after a finished word
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()) or t == max_new_tokens - 1:
                    break
                if repetition_penalty != 1.0:
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(nxt[:, None])
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence_encdec(make_model, vocab_size, src_len=20, tgt_len=12, rtol=1e-4):
    """Your forward vs patched forward vs the generation path (encdec_session: encode once, cached cross-attention
    K/V, one decoder token per step). Returns relative max errors (fp32, CPU)."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    src = torch.randint(0, vocab_size, (2, src_len))
    tgt = torch.randint(0, vocab_size, (2, tgt_len))
    tgt[:, 0] = src[:, -1]                    # session with start_id=None starts from the last source token
    a = ref(src, tgt)
    scale = a.abs().max().item()
    err_full = (a - fast(src, tgt)).abs().max().item() / scale
    with encdec_session(fast, src, tgt_len, torch.float32, start_id=None) as (first, step):
        outs = [first[:, None]]
        for t in range(1, tgt_len):
            outs.append(step(tgt[:, t:t + 1])[:, None])
    err_cache = (a - torch.cat(outs, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache


@torch.no_grad()
def check_equivalence_prefix(make_model, vocab_size, prefix, seq_len=None, rtol=1e-4):
    """Your forward vs patched forward vs (prefill >= prefix tokens, then cached one-token steps)."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    seq_len = seq_len or prefix + 12
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    n_new = seq_len - prefix
    with prefix_session(fast, idx[:, :prefix], n_new, torch.float32) as (first, step):
        outs = [first[:, None]]
        for t in range(prefix, seq_len - 1):
            outs.append(step(idx[:, t:t + 1])[:, None])
    err_cache = (a[:, prefix - 1:seq_len - 1] - torch.cat(outs, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile /kaggle/working/phase2/fetch_data.py
#!/usr/bin/env python3
"""Download Phase_2_LNER/ from the gated Hugging Face dataset (plumbing; declare it).

Token lookup order: Kaggle secret "HF" -> HF_TOKEN environment variable -> hidden prompt.
The token is used for these requests only: never printed, never written to disk.

  python fetch_data.py                      # -> ./data/Phase_2_LNER/...
  python fetch_data.py --out /kaggle/working/data
"""
import argparse
import getpass
import json
import os
import urllib.error
import urllib.request

REPO = "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/"
FILES = [
    "Phase_2_LNER/Train/fold1-text.json",
    "Phase_2_LNER/Train/fold1-gold.json",
    "Phase_2_LNER/Train/fold2-text.json",
    "Phase_2_LNER/Train/fold2-gold.json",
    "Phase_2_LNER/Train/labels.txt",
    "Phase_2_LNER/Test/fold3-text.json",
]


def get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (input hidden): ").strip()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "data"))
    args = ap.parse_args()
    if all(os.path.exists(os.path.join(args.out, rel)) for rel in FILES):
        print("all Phase-2 files already present in", args.out)
        return
    token = get_token()
    try:
        for rel in FILES:
            dst = os.path.join(args.out, rel)
            if os.path.exists(dst):
                print("exists:", dst)
                continue
            req = urllib.request.Request(REPO + rel, headers={"Authorization": f"Bearer {token}"})
            try:
                with urllib.request.urlopen(req, timeout=120) as r:
                    data = r.read()
            except urllib.error.HTTPError as e:
                hint = "check the token / dataset access" if e.code in (401, 403) else e.reason
                raise SystemExit(f"download failed for {rel}: HTTP {e.code} ({hint})")
            if rel.endswith(".json"):
                json.loads(data)                       # validate before saving
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with open(dst, "wb") as f:
                f.write(data)
            print(f"saved {dst} ({len(data) / 1e6:.1f} MB)")
    finally:
        del token


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase2/ner_data.py
"""Phase 2 (L-NER) data loading: documents, gold spans, labels, held-out split (plumbing; declare it).

Internally every span is (start, end, label) with an EXCLUSIVE end, i.e. text[start:end] is the entity.
The spec's own example ([18, 40, "COURT"] on "REPORTABLE IN THE SUPREME COURT OF INDIA") is exclusive;
load_gold() also checks the convention against the entity texts in the gold files and stores it in
END_CONVENTION so predictions are written back in the same convention.
"""
import json
import os
import random
from collections import Counter

END_CONVENTION = {"inclusive": False}      # set by load_gold(); used by to_submission()


def data_root(root):
    """Accepts .../Phase_2_LNER or any parent that contains it."""
    for cand in (root, os.path.join(root, "Phase_2_LNER")):
        if os.path.isdir(os.path.join(cand, "Train")):
            return cand
    raise FileNotFoundError(f"no Phase_2_LNER/Train under {root} (run fetch_data.py first)")


def load_labels(root):
    with open(os.path.join(data_root(root), "Train", "labels.txt"), encoding="utf-8") as f:
        labels = [ln.strip() for ln in f if ln.strip()]
    return labels


def load_texts(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):                    # tolerate [{"id":..., "text":...}, ...]
        data = {str(d.get("id", d.get("doc_id"))): d["text"] for d in data}
    return {str(k): (v["text"] if isinstance(v, dict) else v) for k, v in data.items()}


def _parse_entity(e):
    """-> (start, end, label, text_or_None) from a list/tuple or a dict entry."""
    if isinstance(e, dict):
        s = e.get("start", e.get("start_char", e.get("begin")))
        t = e.get("end", e.get("end_char"))
        lab = e.get("label", e.get("type", e.get("tag")))
        txt = e.get("text", e.get("entity_text", e.get("entity")))
        return int(s), int(t), str(lab), txt
    s, t, lab = e[0], e[1], e[2]
    return int(s), int(t), str(lab), (e[3] if len(e) > 3 else None)


def load_gold(path, texts, labels=None, verbose=True):
    """-> {doc_id: [(start, end_exclusive, label), ...]} sorted by start.
    Detects whether the file's end offsets are exclusive or inclusive from the entity texts."""
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        data = {str(d.get("id", d.get("doc_id"))): d.get("entities", d.get("spans", d.get("labels")))
                for d in data}
    raw = {str(k): [_parse_entity(e) for e in v] for k, v in data.items()}

    excl = incl = with_text = 0
    for doc_id, ents in raw.items():
        text = texts.get(doc_id, "")
        for s, t, _, txt in ents:
            if txt is None:
                continue
            with_text += 1
            excl += text[s:t] == txt
            incl += text[s:t + 1] == txt
    inclusive = with_text > 0 and incl > excl
    END_CONVENTION["inclusive"] = inclusive

    gold, unknown = {}, Counter()
    for doc_id, ents in raw.items():
        spans = []
        for s, t, lab, _ in ents:
            if labels is not None and lab not in labels:
                unknown[lab] += 1
                continue
            spans.append((s, t + 1 if inclusive else t, lab))
        gold[doc_id] = sorted(set(spans))
    if verbose:
        n = sum(len(v) for v in gold.values())
        print(f"{os.path.basename(path)}: {len(gold)} docs, {n} spans | end offsets "
              f"{'INCLUSIVE' if inclusive else 'exclusive'} (text match: exclusive {excl}/{with_text}, "
              f"inclusive {incl}/{with_text})" + (f" | unknown labels dropped: {dict(unknown)}" if unknown else ""))
    missing = [d for d in gold if d not in texts]
    assert not missing, f"{len(missing)} gold docs have no text, e.g. {missing[:3]}"
    return gold


def load_train(root, verbose=True):
    """-> texts {id: str}, gold {id: spans}, fold_of {id: 'fold1'|'fold2'}, labels [..]"""
    root = data_root(root)
    labels = load_labels(root)
    texts, gold, fold_of = {}, {}, {}
    for fold in ("fold1", "fold2"):
        t = load_texts(os.path.join(root, "Train", f"{fold}-text.json"))
        g = load_gold(os.path.join(root, "Train", f"{fold}-gold.json"), t, labels, verbose)
        dup = set(t) & set(texts)
        assert not dup, f"doc ids appear in both folds: {list(dup)[:3]}"
        texts.update(t)
        for d in t:
            gold[d] = g.get(d, [])
            fold_of[d] = fold
    return texts, gold, fold_of, labels


def load_test(root):
    return load_texts(os.path.join(data_root(root), "Test", "fold3-text.json"))


def split_docs(doc_ids, fold_of, split):
    """split = {"mode": "random", "val_frac": float, "seed": int} | {"mode": "fold", "val_fold": "fold2"}
    | {"mode": "none"} (train on everything, no validation).  -> (train_ids, val_ids)"""
    ids = sorted(doc_ids)
    mode = split["mode"]
    if mode == "none":
        return ids, []
    if mode == "fold":
        val = [d for d in ids if fold_of[d] == split["val_fold"]]
        return [d for d in ids if d not in set(val)], val
    if mode == "random":
        rng = random.Random(split["seed"])
        shuffled = ids[:]
        rng.shuffle(shuffled)
        n_val = max(1, round(len(ids) * split["val_frac"]))
        val = sorted(shuffled[:n_val])
        return [d for d in ids if d not in set(val)], val
    raise ValueError(f"unknown split mode {mode}")


def to_submission(pred):
    """{id: [(s, e_exclusive, label)]} -> {id: [[s, e, label]]} in the gold files' end convention."""
    shift = 1 if END_CONVENTION["inclusive"] else 0
    return {d: [[int(s), int(e) - shift, lab] for s, e, lab in spans] for d, spans in pred.items()}


def label_stats(gold, labels):
    c = Counter(lab for spans in gold.values() for _, _, lab in spans)
    return {lab: c.get(lab, 0) for lab in labels}

In [ ]:
%%writefile /kaggle/working/phase2/ner_align.py
"""Character <-> token alignment for the legal ByteBPE, span <-> tag conversion, windows (plumbing; declare it).

tokenize_with_offsets() reproduces ByteBPE.encode() exactly (same whitespace split, same legal pre-tokenisation,
same merges) while recording, for every token, the character span [start, end) it covers in the ORIGINAL text.
No text normalisation is applied, so offsets refer to the raw document. A token that is only the end-of-word
marker </w> (no bytes) is given the span of the last character of its word.

Tag schemes: "BIO" (B-X, I-X) and "BIOES" (adds E-X for the last token, S-X for single-token entities).
"""
import bisect
import re

import numpy as np

from a1_tokenizer import PRETOKENIZE

WORD = re.compile(r"\S+")          # str.split() semantics: maximal runs of non-whitespace


# ---------------------------------------------------------------------------
# tokenisation with character offsets
# ---------------------------------------------------------------------------

PUNCT_RUN = re.compile(r"[^\w\s]{2,}")


def _word_pieces(word, split_punct=False):
    """(piece, start_in_word, is_last) exactly as a1_tokenizer.split_word, with positions.
    split_punct=True additionally splits every punctuation run ("]." -> "]", ".") into single characters, so an
    entity boundary inside a run (e.g. "...SC 1728]." with the entity ending at "]") falls on a token boundary.
    Every single character is in the vocabulary, but the ids then differ from ByteBPE.encode() at such runs."""
    ms = [(m.group(), m.start()) for m in PRETOKENIZE.finditer(word)]
    if not ms:
        return [(word, 0, True)]
    if split_punct:
        ms = [(ch, p0 + j) for piece, p0 in ms
              for j, ch in (enumerate(piece) if PUNCT_RUN.fullmatch(piece) else [(0, piece)])]
    last = len(ms) - 1
    return [(piece, p0, i == last) for i, (piece, p0) in enumerate(ms)]


class OffsetTokenizer:
    def __init__(self, tok, split_punct=False):
        self.tok = tok
        self.split_punct = split_punct
        self._cache = {}

    def _encode_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        tok = self.tok
        ids, st, en = [], [], []
        for piece, p0, is_last in _word_pieces(word, self.split_punct):
            sym = tok._bpe(tok._piece_symbols(piece, is_last))
            cum = [0]
            for ch in piece:
                cum.append(cum[-1] + len(ch.encode("utf-8")))
            b = 0
            for t in sym:
                nb = len(tok.token_bytes[t])
                if nb == 0:                                  # bare </w>: last char of the piece
                    s_c, e_c = len(piece) - 1, len(piece)
                else:
                    s_c = bisect.bisect_right(cum, b) - 1    # char containing the first byte
                    e_c = bisect.bisect_left(cum, b + nb)    # first char boundary at/after the last byte
                    b += nb
                ids.append(t)
                st.append(p0 + s_c)
                en.append(p0 + e_c)
        out = (ids, st, en)
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = out
        return out

    def __call__(self, text):
        """-> ids, starts, ends as int32 numpy arrays (char offsets into `text`, end exclusive)."""
        ids, st, en = [], [], []
        for m in WORD.finditer(text):
            w_ids, w_st, w_en = self._encode_word(m.group())
            off = m.start()
            ids.extend(w_ids)
            st.extend(off + s for s in w_st)
            en.extend(off + e for e in w_en)
        return (np.asarray(ids, dtype=np.int32), np.asarray(st, dtype=np.int32), np.asarray(en, dtype=np.int32))


# ---------------------------------------------------------------------------
# tags
# ---------------------------------------------------------------------------

def tag_list(labels, scheme):
    prefixes = {"BIO": ("B", "I"), "BIOES": ("B", "I", "E", "S")}[scheme]
    return ["O"] + [f"{p}-{lab}" for lab in labels for p in prefixes]


def spans_to_tags(spans, starts, ends, tag2id, scheme):
    """Gold char spans -> one tag id per token. A token belongs to an entity if it overlaps it.
    Returns tags (int64 array) and the number of entities skipped because a token was already taken
    (overlapping/nested gold entities cannot be expressed with one tag per token)."""
    n = len(starts)
    tags = np.zeros(n, dtype=np.int64)
    taken = np.zeros(n, dtype=bool)
    skipped = 0
    for s, e, lab in spans:
        lo = bisect.bisect_right(ends, s)                    # first token with end > s
        hi = bisect.bisect_left(starts, e)                   # first token with start >= e
        if lo >= hi or taken[lo:hi].any():
            skipped += 1
            continue
        taken[lo:hi] = True
        if scheme == "BIOES" and hi - lo == 1:
            tags[lo] = tag2id[f"S-{lab}"]
            continue
        tags[lo] = tag2id[f"B-{lab}"]
        tags[lo + 1:hi] = tag2id[f"I-{lab}"]
        if scheme == "BIOES":
            tags[hi - 1] = tag2id[f"E-{lab}"]
    return tags, skipped


def tags_to_spans(tag_ids, starts, ends, id2tag, repair="lenient"):
    """Token tags -> char spans [(start, end_exclusive, label)].
    repair="lenient": an I-/E- tag that does not continue an open entity of the same label starts a new one.
    repair="strict" : such tags are treated as O."""
    spans = []
    cur_lab, cur_s, cur_e = None, None, None

    def close():
        nonlocal cur_lab
        if cur_lab is not None:
            spans.append((int(cur_s), int(cur_e), cur_lab))
        cur_lab = None

    for i, t in enumerate(tag_ids):
        tag = id2tag[int(t)]
        if tag == "O":
            close()
            continue
        p, lab = tag.split("-", 1)
        if p == "S":
            close()
            spans.append((int(starts[i]), int(ends[i]), lab))
        elif p == "B":
            close()
            cur_lab, cur_s, cur_e = lab, starts[i], ends[i]
        else:                                                 # I or E
            if cur_lab == lab:
                cur_e = ends[i]
            elif repair == "lenient":
                close()
                cur_lab, cur_s, cur_e = lab, starts[i], ends[i]
            else:
                close()
                continue
            if p == "E":
                close()
    close()
    return spans


# ---------------------------------------------------------------------------
# windows
# ---------------------------------------------------------------------------

def window_starts(n, window, stride):
    """Start offsets of windows of `window` tokens every `stride` tokens; the last window is shifted left so it
    ends exactly at n (no padding needed). A document shorter than `window` is one window of length n."""
    if n <= window:
        return [0]
    starts = list(range(0, n - window + 1, stride))
    if starts[-1] != n - window:
        starts.append(n - window)
    return starts


def merge_weights(length, mode):
    """Per-position preference inside one window for overlap merging.
    "center": distance to the nearer window edge (tokens see context on both sides)
    "left_context": position in the window (most left context: natural for causal models)
    "mean": uniform (logits of all covering windows are averaged)"""
    pos = np.arange(length)
    if mode == "center":
        return np.minimum(pos, length - 1 - pos)
    if mode == "left_context":
        return pos
    if mode == "mean":
        return np.zeros(length)
    raise ValueError(f"unknown merge mode {mode}")

In [ ]:
%%writefile /kaggle/working/phase2/ner_metrics.py
"""Strict span metrics for L-NER (plumbing; declare it).

strict match = same start, same end, same label (spec §8.3.5 / §8.4).
Primary: macro-F1 over the 12 labels (per-label F1 averaged; a label with no gold and no predictions is skipped,
a label with gold but no predictions counts as F1 = 0). Also reported: micro P/R/F1 and error counts
(boundary error = right label, overlapping but not identical span; label error = identical span, wrong label).
Counts only: interpreting them is up to you (course rule 9).
"""
from collections import Counter, defaultdict


def _prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def strict_scores(gold, pred, labels):
    """gold, pred: {doc_id: [(start, end, label), ...]} (same end convention). Docs missing from pred = no spans."""
    tp, fp, fn = Counter(), Counter(), Counter()
    boundary, label_err, confusion = Counter(), Counter(), Counter()
    for doc_id, g_spans in gold.items():
        g = set(map(tuple, g_spans))
        p = set(map(tuple, pred.get(doc_id, [])))
        for s in g & p:
            tp[s[2]] += 1
        for s in p - g:
            fp[s[2]] += 1
        for s in g - p:
            fn[s[2]] += 1
        g_by_span = {(s, e): lab for s, e, lab in g}
        for s, e, lab in p - g:
            if (s, e) in g_by_span:
                label_err[lab] += 1
                confusion[(g_by_span[(s, e)], lab)] += 1
            elif any(lab == gl and s < ge and gs < e for gs, ge, gl in g):
                boundary[lab] += 1
    per = {}
    for lab in labels:
        if tp[lab] + fp[lab] + fn[lab] == 0:
            continue
        p, r, f = _prf(tp[lab], fp[lab], fn[lab])
        per[lab] = dict(p=round(p, 4), r=round(r, 4), f1=round(f, 4), tp=tp[lab], fp=fp[lab], fn=fn[lab],
                        boundary_err=boundary[lab], label_err=label_err[lab])
    n = max(1, len(per))
    mp, mr, mf = _prf(sum(tp.values()), sum(fp.values()), sum(fn.values()))
    return {
        "macro_f1": round(sum(v["f1"] for v in per.values()) / n, 4),
        "macro_p": round(sum(v["p"] for v in per.values()) / n, 4),
        "macro_r": round(sum(v["r"] for v in per.values()) / n, 4),
        "micro_p": round(mp, 4), "micro_r": round(mr, 4), "micro_f1": round(mf, 4),
        "per_label": per,
        "confusion_gold_to_pred": {f"{a}->{b}": c for (a, b), c in confusion.most_common()},
    }

In [ ]:
%%writefile /kaggle/working/phase2/backbones.py
"""Phase-1 checkpoints -> token feature extractors for L-NER (plumbing; declare it).

Your Phase-1 model code is used unchanged (phase1_models/: arch1/arch3 for A/B, enc_dec + parts for C,
prefix_lm + prefix_attention for D). The LM head is replaced by nn.Identity, so YOUR forward() returns the
final-LayerNorm hidden states (B, T, d_model) instead of vocabulary logits. Attention runs through the same
fastattn patches as Phase 1 (fused SDPA, identical maths; smoke_test.py re-checks equality).

Backbone options (cfg["backbone"]):
  prefix         D only: "full" -> prefix = window length (every token sees the whole window),
                 int p -> bidirectional over the first p tokens, causal after (your Phase-1 D used 640),
                 0 -> fully causal.
  encdec_mode    C only: "encoder" -> encoder states only (decoder weights dropped; bidirectional),
                 "decoder" -> the window is fed to the encoder AND to the decoder (causal self-attention +
                 cross-attention to the encoder states); features = decoder final states.
  truncate_layers keep only the first k transformer blocks (e.g. a smaller KD student); None = all.
  dropout        override every nn.Dropout p in the backbone; None = keep the checkpoint's value.
"""
import importlib
import os
import sys

import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath(__file__))
for _p in (HERE, os.path.join(HERE, "phase1_models")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import fastattn_ab   # noqa: E402  (A/B attention: arch1 / arch3)
import fastattn_cd   # noqa: E402  (C/D attention: encoder / decoder / cross / prefix)


def _clean_state(sd):
    out = {}
    for k, v in sd.items():
        for pre in ("_orig_mod.", "module."):
            if k.startswith(pre):
                k = k[len(pre):]
        out[k] = v
    return out


def build_lm(spec):
    """spec = the 'config' dict saved inside a Phase-1 best.pt -> (lm, kind)."""
    kind = spec.get("kind", "decoder")
    arch = importlib.import_module(spec["module"])
    if kind == "decoder":
        lm = arch.DecoderLM(arch.LMConfig(**spec["model"]))
    elif kind == "prefix":
        lm = arch.DecoderLM(arch.LMConfig(**spec["model"]))
    elif kind == "encdec":
        lm = arch.EncDecLM(arch.EncLMConfig(**spec["encoder"]), arch.DecLMConfig(**spec["decoder"]))
    else:
        raise ValueError(f"unknown model kind {kind}")
    return lm, kind


def load_phase1(path, load_weights=True):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    spec = ck["config"]
    lm, kind = build_lm(spec)
    if load_weights:
        lm.load_state_dict(_clean_state(ck["model"]))
    info = dict(path=path, kind=kind, module=spec["module"], tag=spec.get("tag"), step=ck.get("step"),
                best_val=ck.get("best_val"))
    return lm, kind, spec, info


class Backbone(nn.Module):
    """forward(ids (B, T) long) -> hidden states (B, T, d_model).
    Attributes for technique code: .lm (your Phase-1 module), .kind, .blocks (list of transformer blocks,
    input -> output order; C "decoder" mode = encoder blocks then decoder blocks), .d_model, .max_len."""

    def __init__(self, lm, kind, prefix=None, encdec_mode=None, truncate_layers=None, dropout=None):
        super().__init__()
        self.kind = kind
        if kind == "encdec":
            assert encdec_mode in ("encoder", "decoder"), "C needs backbone.encdec_mode = 'encoder' | 'decoder'"
            self.encdec_mode = encdec_mode
            if encdec_mode == "encoder":
                for name in ("decblocks", "tok_emb2", "ln_f_dec", "head"):
                    delattr(lm, name)
                lm.ln_f_dec = lm.head = None
            else:
                lm.head = nn.Identity()
            self.d_model = lm.cfg1.d_model
            self.max_len = lm.cfg1.max_seq_len if encdec_mode == "encoder" else min(lm.cfg1.max_seq_len,
                                                                                    lm.cfg2.max_seq_len)
        else:
            if kind == "prefix":
                assert prefix is not None, "D needs backbone.prefix = 'full' | int (0 = causal)"
            self.prefix = prefix
            lm.head = nn.Identity()
            self.d_model = lm.cfg.d_model
            self.max_len = lm.cfg.max_seq_len
        self.lm = lm
        if truncate_layers:
            self._truncate(int(truncate_layers))
        if dropout is not None:
            for m in self.modules():
                if isinstance(m, nn.Dropout):
                    m.p = float(dropout)

    @property
    def blocks(self):
        lm = self.lm
        if self.kind == "encdec":
            return list(lm.encblocks) + (list(lm.decblocks) if self.encdec_mode == "decoder" else [])
        return list(lm.blocks)

    def _truncate(self, k):
        lm = self.lm
        if self.kind == "encdec":
            assert self.encdec_mode == "encoder", "truncate_layers for C is only supported in encoder mode"
            lm.encblocks = nn.ModuleList(list(lm.encblocks)[:k])
        else:
            lm.blocks = nn.ModuleList(list(lm.blocks)[:k])

    def forward(self, ids):
        lm = self.lm
        if self.kind == "decoder":
            return lm(ids)
        if self.kind == "prefix":
            lm.cfg.prefix = ids.shape[1] if self.prefix == "full" else int(self.prefix)
            return lm(ids)
        memory = lm.encode(ids)
        if self.encdec_mode == "encoder":
            return memory
        return lm.decode(ids, memory)


def patch_fast_attention(lm, kind):
    if kind == "decoder":
        fastattn_ab.patch_model(lm)
    else:
        fastattn_cd.patch_model(lm)
    return lm


def build_backbone(ckpt_path, bcfg, vocab_size=None, load_weights=True, fast_attention=True):
    lm, kind, spec, info = load_phase1(ckpt_path, load_weights)
    if vocab_size is not None:
        v = spec["model"]["vocab_size"] if "model" in spec else spec["encoder"]["vocab_size"]
        assert v == vocab_size, f"{ckpt_path}: vocab {v} != tokenizer vocab {vocab_size} (different tokenizer?)"
    if fast_attention:
        patch_fast_attention(lm, kind)
    bb = Backbone(lm, kind, prefix=bcfg.get("prefix"), encdec_mode=bcfg.get("encdec_mode"),
                  truncate_layers=bcfg.get("truncate_layers"), dropout=bcfg.get("dropout"))
    return bb, spec, info

In [ ]:
%%writefile /kaggle/working/phase2/inject.py
"""Generic module-surgery helpers for your fine-tuning techniques (plumbing; declare it).

These only find / replace / freeze modules; they contain no adaptation method. Typical use in your files:

    from inject import freeze, replace_linears, wrap_modules
    freeze(backbone)                                            # base weights frozen
    replace_linears(backbone, lambda name, lin: name.endswith(("q_proj", "v_proj")),
                    lambda name, lin: YourWrapper(lin, ...))    # your LoRA / LoRMA module around nn.Linear
    wrap_modules(backbone, lambda name, m: name.endswith(".ffn"),
                 lambda name, m: YourAdapterWrapper(m, ...))    # your adapter after a sublayer

Linear layer names in the Phase-1 models (relative to backbone, e.g. "lm.blocks.3.attn.q_proj"):
  A/B/D blocks:  attn.q_proj, attn.k_proj, attn.v_proj, attn.out_proj, ffn.net.0 (d->d_ff), ffn.net.2 (d_ff->d)
  C encoder:     lm.encblocks.<i>.attn.{q,k,v,out}_proj, lm.encblocks.<i>.ffn.net.{0,2}
  C decoder:     lm.decblocks.<i>.attn.*, lm.decblocks.<i>.crattn.{q,k,v,out}_proj, lm.decblocks.<i>.ffn.net.{0,2}
Attention projections have no bias (bias=False); FFN linears and LayerNorms do.
Sublayer call signatures: attn(x) (D: attn(x, p)), crattn(x, memory), ffn(x) -> a wrapper should take
(*args, **kwargs) and pass them through to the wrapped module.
"""
import torch.nn as nn


def freeze(module):
    for p in module.parameters():
        p.requires_grad_(False)
    return module


def unfreeze(module):
    for p in module.parameters():
        p.requires_grad_(True)
    return module


def _set_child(root, name, new):
    parent_name, _, child = name.rpartition(".")
    parent = root.get_submodule(parent_name) if parent_name else root
    setattr(parent, child, new)


def list_linears(root):
    return [name for name, m in root.named_modules() if isinstance(m, nn.Linear)]


def replace_linears(root, match, factory):
    """Replace every nn.Linear for which match(name, linear) is True by factory(name, linear). -> replaced names."""
    targets = [(n, m) for n, m in root.named_modules() if isinstance(m, nn.Linear) and match(n, m)]
    for name, lin in targets:
        _set_child(root, name, factory(name, lin))
    return [n for n, _ in targets]


def wrap_modules(root, match, factory):
    """Replace every submodule for which match(name, module) is True by factory(name, module). -> replaced names.
    Matches are collected first, so a wrapper containing the original module is not matched again."""
    targets = [(n, m) for n, m in root.named_modules() if n and match(n, m)]
    for name, mod in targets:
        _set_child(root, name, factory(name, mod))
    return [n for n, _ in targets]


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_pct": round(100 * trainable / max(1, total), 4)}

In [ ]:
%%writefile /kaggle/working/phase2/techniques/__init__.py
"""Technique registry (plumbing). The config's technique.name selects techniques/<name>.py.

Contract for every technique file (YOUR code, course rule 9):
    apply(backbone, **cfg) -> None
        Called once, after the Phase-1 weights are loaded and before the optimizer is built. On entry EVERY backbone
        parameter has requires_grad=True. On exit, requires_grad must be True exactly on what should train
        (plus any modules you inserted). The head is separate and always trained.
        backbone: backbones.Backbone (.lm = your Phase-1 module, .blocks, .kind, .d_model). See inject.py for
        freeze / replace_linears / wrap_modules helpers and the linear-layer names.
    optional: param_groups(model, base_lr) -> list of optimizer param-group dicts (e.g. a different lr for your
        inserted modules). If absent, every trainable backbone parameter uses train.lr and the head uses
        train.head_lr (or train.lr).
"""
import importlib


def get(name):
    return importlib.import_module(f"techniques.{name}")

In [ ]:
%%writefile /kaggle/working/phase2/techniques/full_ft.py
"""Full fine-tuning: every backbone parameter trains (plumbing-level: nothing to implement)."""


def apply(backbone, **cfg):
    assert not cfg, f"full_ft takes no options, got {cfg}"
    for p in backbone.parameters():
        p.requires_grad_(True)

In [ ]:
%%writefile /kaggle/working/phase2/train_ner.py
#!/usr/bin/env python3
"""Fine-tune one Phase-1 model for L-NER and write its Codabench submission (plumbing; declare it).

  python train_ner.py run_config.json

Pipeline: load docs + gold -> tokenize with char offsets (ner_align) -> gold spans to token tags -> sliding windows
-> Phase-1 backbone (backbones) + YOUR technique (techniques/<name>.py) + YOUR head (heads.py)
-> AdamW training (fp16 autocast on GPU) with strict macro-F1 on the held-out docs after every epoch
-> best checkpoint -> val report -> test predictions -> prediction.json + submission_<name>.zip.

Outputs in <out_dir>/<name>/:
  config.json, log.jsonl, best_model.pt (full state, fp32), val_scores.json,
  val_predictions.json, prediction.json, submission_<name>.zip, summary.json (scores, params, memory, timings).

Config keys whose value is None in the template are your choices (course rule 9) and must be filled; see
configs/template.json. Everything marked "plumbing" has a fixed meaning and does not change results.
"""
import json
import math
import os
import random
import sys
import time
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

import backbones                      # noqa: E402
import heads                          # noqa: E402
import ner_align                      # noqa: E402
import ner_data                       # noqa: E402
import techniques                     # noqa: E402
from a1_tokenizer import ByteBPE      # noqa: E402
from inject import count_params       # noqa: E402
from ner_metrics import strict_scores  # noqa: E402

IGNORE = -100

# keys that must be set (not None); KD / backbone-specific keys are checked separately
REQUIRED = {
    "preprocess": ["split_punct", "tag_scheme"],
    "window": ["train_len", "train_stride", "infer_len", "infer_stride", "merge"],
    "train": ["epochs", "batch_size", "micro_batch", "lr", "weight_decay", "warmup_frac", "schedule", "grad_clip"],
    "decode": ["repair"],
}


def check_config(cfg):
    missing = [f"{sec}.{k}" for sec, keys in REQUIRED.items() for k in keys if cfg.get(sec, {}).get(k) is None]
    if cfg.get("head", {}).get("name") is None:
        missing.append("head.name")
    if cfg.get("technique", {}).get("name") is None:
        missing.append("technique.name")
    if cfg["split"]["mode"] == "random" and cfg["split"].get("val_frac") is None:
        missing.append("split.val_frac")
    assert not missing, "fill these config values first: " + ", ".join(missing)
    check_technique_options(cfg)
    tr = cfg["train"]
    assert tr["batch_size"] % tr["micro_batch"] == 0, "train.batch_size must be a multiple of train.micro_batch"
    assert cfg["preprocess"]["tag_scheme"] in ("BIO", "BIOES")
    assert cfg["train"]["schedule"] in ("linear", "cosine", "constant")
    assert cfg["decode"]["repair"] in ("lenient", "strict")


def check_technique_options(cfg):
    """Fail before any GPU time is spent if the technique options do not fit YOUR apply() signature
    (unknown key such as "rank" instead of "r", or a missing required option)."""
    import inspect
    tech = cfg["technique"]
    opts = {k: v for k, v in tech.items() if k != "name"}
    fn = techniques.get(tech["name"]).apply
    sig = inspect.signature(fn)
    try:
        sig.bind(None, **opts)                       # None stands in for the backbone argument
    except TypeError as e:
        raise AssertionError(f"technique '{tech['name']}' options {opts} do not fit apply{sig}: {e}")


class Logger:
    def __init__(self, path):
        self.path = path

    def __call__(self, msg, **kv):
        rec = {"t": round(time.time(), 1), "msg": msg, **kv}
        print(json.dumps(rec), flush=True)
        with open(self.path, "a") as f:
            f.write(json.dumps(rec) + "\n")


# ---------------------------------------------------------------------------
# model
# ---------------------------------------------------------------------------

class NERModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, ids):
        return self.head(self.backbone(ids))


def build_model(cfg, n_tags, vocab_size, load_weights=True, apply_technique=True):
    bb, spec, info = backbones.build_backbone(cfg["phase1_ckpt"], cfg.get("backbone", {}), vocab_size,
                                              load_weights=load_weights,
                                              fast_attention=cfg.get("fast_attention", True))
    for p in bb.parameters():
        p.requires_grad_(True)
    tech = cfg["technique"]
    tmod = techniques.get(tech["name"])
    if apply_technique:
        tmod.apply(bb, **{k: v for k, v in tech.items() if k != "name"})
    head = heads.build_head(bb.d_model, n_tags, **cfg["head"])
    for p in head.parameters():
        p.requires_grad_(True)
    return NERModel(bb, head), tmod, info


def save_model(model, path):
    # full precision for every tensor: reloading must reproduce the evaluated model exactly
    torch.save(model.state_dict(), path + ".tmp")
    os.replace(path + ".tmp", path)


def load_run(run_dir, device, vocab_size):
    """Rebuild a finished run's model (e.g. a KD teacher) from its config.json + best_model.pt."""
    cfg = json.load(open(os.path.join(run_dir, "config.json")))
    tags = json.load(open(os.path.join(run_dir, "tags.json")))
    model, _, _ = build_model(cfg, len(tags), vocab_size, load_weights=False)
    sd = torch.load(os.path.join(run_dir, "best_model.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
    return model.to(device).eval(), cfg, tags


# ---------------------------------------------------------------------------
# data
# ---------------------------------------------------------------------------

def encode_docs(texts, gold, otok, tag2id, scheme):
    out, skipped = {}, 0
    for d, text in texts.items():
        ids, st, en = otok(text)
        if gold is not None:
            tags, sk = ner_align.spans_to_tags(gold.get(d, []), st, en, tag2id, scheme)
            skipped += sk
        else:
            tags = None
        out[d] = dict(ids=ids, starts=st, ends=en, tags=tags)
    return out, skipped


def make_windows(enc, doc_ids, length, stride):
    """-> list of (doc_id, start, window_length)."""
    wins = []
    for d in doc_ids:
        n = len(enc[d]["ids"])
        if n == 0:                                   # empty document (fold3 has one): no windows, predicts []
            continue
        for s in ner_align.window_starts(n, length, stride):
            wins.append((d, s, min(length, n)))
    return wins


def batches(wins, micro, shuffle, rng):
    """Group windows of equal length into micro-batches (no padding anywhere)."""
    by_len = {}
    for w in wins:
        by_len.setdefault(w[2], []).append(w)
    out = []
    for group in by_len.values():
        if shuffle:
            rng.shuffle(group)
        out += [group[i:i + micro] for i in range(0, len(group), micro)]
    if shuffle:
        rng.shuffle(out)
    return out


def to_tensors(enc, mb, device, with_tags=True):
    ids = torch.tensor(np.stack([enc[d]["ids"][s:s + n] for d, s, n in mb]), dtype=torch.long, device=device)
    if not with_tags:
        return ids, None
    tags = torch.tensor(np.stack([enc[d]["tags"][s:s + n] for d, s, n in mb]), dtype=torch.long, device=device)
    return ids, tags


def class_weights(enc, doc_ids, n_tags, mode, o_weight):
    if mode is None and o_weight is None:
        return None
    w = torch.ones(n_tags)
    if mode is not None:
        counts = np.bincount(np.concatenate([enc[d]["tags"] for d in doc_ids]), minlength=n_tags).astype(np.float64)
        counts = np.maximum(counts, 1.0)
        power = {"inv_freq": 1.0, "inv_sqrt": 0.5}[mode]
        w = torch.tensor((counts.sum() / counts) ** power, dtype=torch.float32)
        w = w / w.mean()
    if o_weight is not None:
        w[0] = float(o_weight)
    return w


# ---------------------------------------------------------------------------
# inference
# ---------------------------------------------------------------------------

@torch.no_grad()
def doc_logits(model, enc, doc_ids, length, stride, merge, micro, device, amp):
    """Merged per-token logits for every doc: {doc_id: (N, n_tags) float32 tensor on CPU}."""
    model.eval()
    wins = make_windows(enc, doc_ids, length, stride)
    best = {d: np.full(len(enc[d]["ids"]), -1.0) for d in doc_ids}
    sums = {}
    counts = {d: np.zeros(len(enc[d]["ids"])) for d in doc_ids}
    for mb in batches(wins, micro, False, None):
        ids, _ = to_tensors(enc, mb, device, with_tags=False)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            logits = model(ids).float().cpu()
        for (d, s, n), lg in zip(mb, logits):
            if d not in sums:
                sums[d] = torch.zeros(len(enc[d]["ids"]), lg.shape[-1])
            if merge == "mean":
                sums[d][s:s + n] += lg
                counts[d][s:s + n] += 1
            else:
                w = ner_align.merge_weights(n, merge)
                upd = w > best[d][s:s + n]
                idx = torch.from_numpy(np.nonzero(upd)[0])
                sums[d][s + idx] = lg[idx]
                best[d][s + idx] = w[idx]
    if merge == "mean":
        for d in sums:
            sums[d] /= torch.tensor(counts[d], dtype=torch.float32).unsqueeze(-1)
    return sums


@torch.no_grad()
def predict_spans(model, enc, doc_ids, cfg, id2tag, device, amp):
    w = cfg["window"]
    logits = doc_logits(model, enc, doc_ids, w["infer_len"], w["infer_stride"], w["merge"],
                        cfg["train"].get("infer_micro_batch") or cfg["train"]["micro_batch"] * 2, device, amp)
    pred = {}
    dec = getattr(model.head, "decode", None)
    for d in doc_ids:
        if len(enc[d]["ids"]) == 0:
            pred[d] = []
            continue
        lg = logits[d]
        if dec is not None:
            tag_ids = dec(lg.unsqueeze(0).to(device)).squeeze(0).cpu().numpy()
        else:
            tag_ids = lg.argmax(-1).numpy()
        pred[d] = ner_align.tags_to_spans(tag_ids, enc[d]["starts"], enc[d]["ends"], id2tag, cfg["decode"]["repair"])
    return pred


# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------

def lr_factor(step, total, warmup, schedule):
    if warmup and step < warmup:
        return (step + 1) / warmup
    if schedule == "constant":
        return 1.0
    prog = min(1.0, (step - warmup) / max(1, total - warmup))
    return 1.0 - prog if schedule == "linear" else 0.5 * (1 + math.cos(math.pi * prog))


def main(cfg_path):
    cfg = json.load(open(cfg_path))
    check_config(cfg)
    name = cfg["name"]
    out = os.path.join(cfg["out_dir"], name)
    os.makedirs(out, exist_ok=True)
    log = Logger(os.path.join(out, "log.jsonl"))
    json.dump(cfg, open(os.path.join(out, "config.json"), "w"), indent=2)
    tr = cfg["train"]
    seed = tr.get("seed", 0)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    t_start = time.time()
    deadline = t_start + 60 * tr["max_minutes"] if tr.get("max_minutes") else None

    # ---- data
    tok = ByteBPE.from_json(cfg["tokenizer"])
    texts, gold, fold_of, labels = ner_data.load_train(cfg["data_dir"], verbose=False)
    test_texts = ner_data.load_test(cfg["data_dir"])
    scheme = cfg["preprocess"]["tag_scheme"]
    tags = ner_align.tag_list(labels, scheme)
    tag2id, id2tag = {t: i for i, t in enumerate(tags)}, dict(enumerate(tags))
    json.dump(tags, open(os.path.join(out, "tags.json"), "w"))
    otok = ner_align.OffsetTokenizer(tok, split_punct=cfg["preprocess"]["split_punct"])
    enc, skipped = encode_docs(texts, gold, otok, tag2id, scheme)
    enc_test, _ = encode_docs(test_texts, None, otok, tag2id, scheme)
    train_ids, val_ids = ner_data.split_docs(list(texts), fold_of, cfg["split"])
    log("data", train_docs=len(train_ids), val_docs=len(val_ids), test_docs=len(test_texts),
        train_tokens=int(sum(len(enc[d]["ids"]) for d in train_ids)), overlap_skipped=skipped, n_tags=len(tags))

    # ---- model
    model, tmod, info = build_model(cfg, len(tags), tok.get_vocab_size())
    model.to(device)
    w = cfg["window"]
    assert max(w["train_len"], w["infer_len"]) <= model.backbone.max_len, \
        f"window longer than the backbone's max_seq_len {model.backbone.max_len}"
    params = count_params(model)
    params["backbone_total"] = sum(p.numel() for p in model.backbone.parameters())
    params["head"] = sum(p.numel() for p in model.head.parameters())
    log("model", **info, **params)

    teacher = None
    if cfg.get("kd"):
        kd = dict(cfg["kd"])
        teacher, tcfg, ttags = load_run(kd.pop("teacher_run"), device, tok.get_vocab_size())
        assert ttags == tags, "teacher and student must use the same labels and tag scheme"
        assert w["train_len"] <= teacher.backbone.max_len, "train window longer than the teacher's max_seq_len"
        for p in teacher.parameters():
            p.requires_grad_(False)
        kd_loss = techniques.get("kd").kd_loss
        log("teacher", run=tcfg["name"], params=count_params(teacher)["total"])

    # ---- optimizer
    if hasattr(tmod, "param_groups"):
        groups = tmod.param_groups(model, tr["lr"])
    else:
        bb_params = [p for p in model.backbone.parameters() if p.requires_grad]
        groups = [{"params": bb_params, "lr": tr["lr"]},
                  {"params": list(model.head.parameters()), "lr": tr.get("head_lr") or tr["lr"]}]
    for g in groups:
        g["params"] = list(g["params"])
        g["base_lr"] = g["lr"]
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=tr["weight_decay"], betas=tuple(tr.get("betas", (0.9, 0.999))))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    cw = class_weights(enc, train_ids, len(tags), tr.get("class_weights"), tr.get("o_weight"))
    if cw is not None:
        cw = cw.to(device)
        log("class_weights", weights={t: round(float(x), 3) for t, x in zip(tags, cw)})

    train_wins = make_windows(enc, train_ids, w["train_len"], w["train_stride"])
    rng = random.Random(seed)
    micro, accum = tr["micro_batch"], tr["batch_size"] // tr["micro_batch"]
    steps_per_epoch = math.ceil(len(batches(train_wins, micro, False, None)) / accum)
    total_steps = steps_per_epoch * tr["epochs"]
    warmup = int(tr["warmup_frac"] * total_steps)
    log("schedule", train_windows=len(train_wins), steps_per_epoch=steps_per_epoch, total_steps=total_steps,
        warmup_steps=warmup)

    best_f1, best_epoch, bad_epochs, step = -1.0, -1, 0, 0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t_train = time.time()
    tokens_seen = 0
    stop = False
    for epoch in range(1, tr["epochs"] + 1):
        model.train()
        if teacher is not None:
            teacher.eval()
        mbs = batches(train_wins, micro, True, rng)
        run_loss, n_loss, t_ep = 0.0, 0, time.time()
        for i in range(0, len(mbs), accum):
            group = mbs[i:i + accum]
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_factor(step, total_steps, warmup, tr["schedule"])
            for mb in group:
                ids, tg = to_tensors(enc, mb, device)
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    logits = model(ids)
                    if teacher is not None:
                        with torch.no_grad():
                            t_logits = teacher(ids)
                logits = logits.float()
                if teacher is not None:
                    loss = kd_loss(logits, t_logits.float(), tg, **kd)
                elif hasattr(model.head, "loss"):
                    loss = model.head.loss(logits, tg)
                else:
                    loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tg.reshape(-1),
                                           weight=cw, ignore_index=IGNORE)
                scaler.scale(loss / len(group)).backward()
                run_loss += loss.item()
                n_loss += 1
                tokens_seen += ids.numel()
            if tr["grad_clip"]:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], tr["grad_clip"])
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step % tr.get("log_every", 20) == 0:
                log("train", epoch=epoch, step=step, loss=round(run_loss / max(1, n_loss), 4),
                    lr=opt.param_groups[0]["lr"], tok_s=round(tokens_seen / (time.time() - t_train)))
            if deadline and time.time() > deadline:
                log("deadline reached: stopping training", step=step)
                stop = True
                break
        ep_loss = run_loss / max(1, n_loss)
        rec = dict(epoch=epoch, step=step, train_loss=round(ep_loss, 4), epoch_s=round(time.time() - t_ep, 1))
        if val_ids and (epoch % tr.get("eval_every_epochs", 1) == 0 or epoch == tr["epochs"] or stop):
            t_ev = time.time()
            sc = strict_scores({d: gold[d] for d in val_ids},
                               predict_spans(model, enc, val_ids, cfg, id2tag, device, amp), labels)
            rec.update(val_macro_f1=sc["macro_f1"], val_micro_f1=sc["micro_f1"], eval_s=round(time.time() - t_ev, 1))
            if sc["macro_f1"] > best_f1:
                best_f1, best_epoch, bad_epochs = sc["macro_f1"], epoch, 0
                save_model(model, os.path.join(out, "best_model.pt"))
                rec["saved"] = True
            else:
                bad_epochs += 1
        log("epoch", **rec)
        if not val_ids:
            save_model(model, os.path.join(out, "best_model.pt"))
            best_epoch = epoch
        if stop or (tr.get("patience") and bad_epochs >= tr["patience"]):
            break
    train_s = time.time() - t_train
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None

    # ---- best model -> val report + test predictions
    sd = torch.load(os.path.join(out, "best_model.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
    summary = dict(name=name, phase1=info, technique=cfg["technique"], head=cfg["head"], kd=bool(cfg.get("kd")),
                   params=params, best_epoch=best_epoch, train_seconds=round(train_s, 1),
                   train_tokens=tokens_seen, peak_mem_GB=round(peak_gb, 2) if peak_gb else None)
    if val_ids:
        val_pred = predict_spans(model, enc, val_ids, cfg, id2tag, device, amp)
        val_sc = strict_scores({d: gold[d] for d in val_ids}, val_pred, labels)
        json.dump(val_sc, open(os.path.join(out, "val_scores.json"), "w"), indent=2)
        json.dump(ner_data.to_submission(val_pred), open(os.path.join(out, "val_predictions.json"), "w"))
        summary.update(val_macro_f1=val_sc["macro_f1"], val_macro_p=val_sc["macro_p"], val_macro_r=val_sc["macro_r"],
                       val_micro_f1=val_sc["micro_f1"])
    t_inf = time.time()
    test_pred = predict_spans(model, enc_test, list(test_texts), cfg, id2tag, device, amp)
    inf_s = time.time() - t_inf
    n_test_tok = sum(len(enc_test[d]["ids"]) for d in test_texts)
    summary.update(test_infer_seconds=round(inf_s, 1), test_infer_tok_s=round(n_test_tok / max(inf_s, 1e-9)),
                   test_spans=sum(len(v) for v in test_pred.values()))
    sub = ner_data.to_submission(test_pred)
    for d in test_texts:
        sub.setdefault(d, [])
    pred_path = os.path.join(out, "prediction.json")
    json.dump(sub, open(pred_path, "w"))
    zip_path = os.path.join(out, f"submission_{name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    summary["total_seconds"] = round(time.time() - t_start, 1)
    json.dump(summary, open(os.path.join(out, "summary.json"), "w"), indent=2)
    log("done", **{k: v for k, v in summary.items() if k not in ("phase1",)}, zip=zip_path)


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/phase2/run_many.py
#!/usr/bin/env python3
"""Run several train_ner.py configs over the available GPUs IN PARALLEL (plumbing; declare it).

  python run_many.py --gpus 0,1 configs/B_full.json configs/C_full.json ...

One worker per GPU; every worker runs one training process at a time on its own GPU (CUDA_VISIBLE_DEVICES), so with
--gpus 0,1 two fine-tuning runs train simultaneously, one per T4. Scheduling uses a SHARED queue: whenever a GPU becomes
free it takes the first runnable config in list order (no GPU sits idle while work remains). A config with "kd" is only
runnable once its teacher has written summary.json; until then the free GPU takes the next job instead. If the teacher
fails (or is not part of this batch and has no summary.json), the KD run is marked FAILED instead of waiting forever.

Logs: <out_dir>/<name>.stdout (+ <out_dir>/<name>/log.jsonl). A status line every --every seconds, a summary at the end.
Exit code 1 if any run failed.
"""
import argparse
import json
import os
import subprocess
import sys
import threading
import time

HERE = os.path.dirname(os.path.abspath(__file__))


class Scheduler:
    def __init__(self, cfg_paths):
        self.lock = threading.Lock()
        self.cfgs = {}
        for p in cfg_paths:
            c = json.load(open(p))
            self.cfgs[c["name"]] = (p, c)
        self.pending = list(self.cfgs)                        # names, in the order given
        self.status = {n: "queued" for n in self.pending}
        self.started = {}

    def _teacher(self, cfg):
        if not cfg.get("kd"):
            return None, None
        d = cfg["kd"]["teacher_run"].rstrip("/")
        return os.path.basename(d), os.path.join(d, "summary.json")

    def next_job(self):
        """-> (name, path, cfg) to run now | "wait" (only KD runs whose teacher is still training) | None (nothing left)."""
        with self.lock:
            if not self.pending:
                return None
            for name in list(self.pending):
                path, cfg = self.cfgs[name]
                teacher, done = self._teacher(cfg)
                if teacher is not None and not os.path.exists(done):
                    t_status = self.status.get(teacher)
                    if t_status is None or t_status.startswith(("FAILED", "done")):
                        self.pending.remove(name)
                        self.status[name] = f"FAILED (teacher {teacher} has no summary.json)"
                        continue
                    self.status[name] = f"waiting for teacher {teacher}"
                    continue
                self.pending.remove(name)
                return name, path, cfg
            return "wait" if self.pending else None

    def worker(self, gpu):
        while True:
            job = self.next_job()
            if job is None:
                return
            if job == "wait":
                time.sleep(10)
                continue
            name, path, cfg = job
            os.makedirs(cfg["out_dir"], exist_ok=True)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2")
            with self.lock:
                self.status[name] = f"running on GPU {gpu}"
                self.started[name] = time.time()
            with open(os.path.join(cfg["out_dir"], f"{name}.stdout"), "w") as log:
                rc = subprocess.call([sys.executable, os.path.join(HERE, "train_ner.py"), path], env=env,
                                     stdout=log, stderr=subprocess.STDOUT, cwd=os.getcwd())
            mins = (time.time() - self.started[name]) / 60
            with self.lock:
                self.status[name] = f"done on GPU {gpu} in {mins:.1f} min" if rc == 0 else \
                    f"FAILED (exit {rc}) on GPU {gpu} after {mins:.1f} min"


def last_log_line(cfg):
    logf = os.path.join(cfg["out_dir"], cfg["name"], "log.jsonl")
    if not os.path.exists(logf):
        return None
    lines = open(logf).read().strip().splitlines()
    return lines[-1][:250] if lines else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("configs", nargs="+")
    ap.add_argument("--gpus", default="0")
    ap.add_argument("--every", type=int, default=300)
    args = ap.parse_args()
    gpus = [g for g in args.gpus.split(",") if g != ""]
    sched = Scheduler(args.configs)
    print(f"{len(sched.cfgs)} runs on GPUs {gpus} ({len(gpus)} in parallel): {list(sched.cfgs)}", flush=True)
    threads = [threading.Thread(target=sched.worker, args=(g,), daemon=True, name=f"gpu{g}") for g in gpus]
    for t in threads:
        t.start()
    t0 = time.time()
    while any(t.is_alive() for t in threads):
        for t in threads:
            t.join(timeout=args.every / len(threads))
        with sched.lock:
            status = dict(sched.status)
        print(f"\n{time.strftime('%H:%M:%S')} (+{(time.time() - t0) / 60:.1f} min)", flush=True)
        for name, st in status.items():
            line = last_log_line(sched.cfgs[name][1]) if st.startswith("running") else None
            print(f"  {name:<20} {st}" + (f"\n      {line}" if line else ""), flush=True)

    print("\nsummary")
    failed = []
    for name, (path, cfg) in sched.cfgs.items():
        sp = os.path.join(cfg["out_dir"], name, "summary.json")
        if os.path.exists(sp):
            s = json.load(open(sp))
            print(f"  {name:<20} val macro-F1 {s.get('val_macro_f1')}  trainable {s['params']['trainable_pct']}%  "
                  f"train {s['train_seconds'] / 60:.1f} min  peak {s['peak_mem_GB']} GB  ({sched.status[name]})")
        else:
            print(f"  {name:<20} no summary ({sched.status[name]})")
        if not sched.status[name].startswith("done"):
            failed.append(name)
    if failed:
        sys.exit(f"failed runs: {failed} (see <out_dir>/<name>.stdout)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase2/check_pipeline.py
#!/usr/bin/env python3
"""Oracle check of the whole prediction path (plumbing; declare it). No model involved.

Gold spans -> token tags -> one-hot "logits" per window -> the SAME window merging, decoding, offset conversion
and submission formatting as train_ner.py -> strict macro-F1 against gold. Any bug in windows/merge/offsets shows up
as a score below the alignment ceiling. Run it for the settings you intend to use:

  python check_pipeline.py --data data --tokenizer ../phase1/ab_inputs/legal_bpe.json
"""
import argparse
import itertools
import json
import os
import sys

import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ner_align                     # noqa: E402
import ner_data                      # noqa: E402
import train_ner                     # noqa: E402
from a1_tokenizer import ByteBPE     # noqa: E402
from ner_metrics import strict_scores  # noqa: E402


class OracleModel(nn.Module):
    """Returns one-hot logits of the gold tags for each window (looked up by the window's token ids)."""

    def __init__(self, enc, doc_ids, length, stride, n_tags):
        super().__init__()
        self.table = {}
        for d, s, n in train_ner.make_windows(enc, doc_ids, length, stride):
            self.table[enc[d]["ids"][s:s + n].tobytes()] = enc[d]["tags"][s:s + n]
        self.n_tags = n_tags
        self.head = nn.Identity()

    def forward(self, ids):
        rows = [self.table[r.astype(np.int32).tobytes()] for r in ids.cpu().numpy()]
        return torch.nn.functional.one_hot(torch.tensor(np.stack(rows)), self.n_tags).float() * 10


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--len", type=int, default=512)
    ap.add_argument("--stride", type=int, default=256)
    args = ap.parse_args()
    tok = ByteBPE.from_json(args.tokenizer)
    texts, gold, fold_of, labels = ner_data.load_train(args.data)
    docs = sorted(texts)
    for split_punct, scheme, merge, repair in itertools.product((False, True), ("BIO", "BIOES"),
                                                                 ("center", "left_context", "mean"), ("lenient",)):
        tags = ner_align.tag_list(labels, scheme)
        t2i, i2t = {t: i for i, t in enumerate(tags)}, dict(enumerate(tags))
        enc, _ = train_ner.encode_docs(texts, gold, ner_align.OffsetTokenizer(tok, split_punct), t2i, scheme)
        model = OracleModel(enc, docs, args.len, args.stride, len(tags))
        cfg = {"window": {"infer_len": args.len, "infer_stride": args.stride, "merge": merge},
               "train": {"micro_batch": 8}, "decode": {"repair": repair}}
        pred = train_ner.predict_spans(model, enc, docs, cfg, i2t, torch.device("cpu"), False)
        # through the submission format and back, exactly as Codabench reads it
        sub = json.loads(json.dumps(ner_data.to_submission(pred)))
        back = {d: [tuple(x[:3]) for x in v] for d, v in sub.items()}
        sc = strict_scores(gold, back, labels)
        print(f"split_punct={split_punct!s:<5} {scheme:<5} merge={merge:<12} macro-F1 {sc['macro_f1']:.4f}  "
              f"micro-F1 {sc['micro_f1']:.4f}")


if __name__ == "__main__":
    main()

## Stage 1b — YOUR files: head + fine-tuning techniques (contracts in each docstring)

In [ ]:
%%writefile /kaggle/working/phase2/heads.py
"""YOUR FILE: prediction heads for L-NER (course rule 9: model design is yours; plumbing only calls this).

Contract used by train_ner.py
-----------------------------
build_head(d_model, n_tags, name, **cfg) -> nn.Module    called with the config's "head" dict as keywords,
                                                           e.g. {"name": "linear"} or {"name": "custom", "hidden": ...}

head.forward(h) -> logits                                  h: (B, T, d_model) backbone states, logits: (B, T, n_tags)

Optional methods (used automatically when present):
head.loss(logits, tags) -> scalar      tags: (B, T) long, -100 = ignore. If absent: (class-weighted) cross-entropy.
                                       Implement this for e.g. a CRF head.
head.decode(logits) -> tag ids         logits: (1, N, n_tags) for one whole document (windows already merged),
                                       returns (1, N) long. If absent: argmax.

Tag ids: 0 = "O", then for each label in labels.txt order: B-, I- (BIO) or B-, I-, E-, S- (BIOES).
The head is always trained (requires_grad=True), whatever the technique.
"""
import torch
import torch.nn as nn
from torchcrf import CRF
class Linear_head(nn.Module):
    def __init__(self,d_model,n_tags,**cfg):
        super().__init__()
        self.in_features = d_model
        self.out_features = n_tags
        self.p = cfg.get("dropout",0.0)
        self.linear = nn.Linear(self.in_features,self.out_features)
        self.dropout = nn.Dropout(p=self.p)
    def forward(self,x:torch.Tensor)->torch.Tensor:
        B,T,C = x.shape
        x = self.dropout(x)
        logits = self.linear(x)
        return logits

class Mlp_head(nn.Module):
    def __init__(self,d_model,n_tags,**cfg):
        super().__init__()
        self.in_features = d_model
        self.out_features = n_tags
        self.hidden = cfg.get("hidden",256)
        self.activation = nn.GELU()
        self.p = cfg.get("dropout",0.0)
        self.linear1 = nn.Linear(self.in_features,self.hidden)
        self.linear2 = nn.Linear(self.hidden,self.out_features)
        self.dropout = nn.Dropout(p=self.p)
    def forward(self,x:torch.Tensor)->torch.Tensor:
        B,T,C = x.shape
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        logits = self.linear2(x)
        return logits

class Mlp_Crf__head(nn.Module):
    def __init__(self,d_model,n_tags,**cfg):
        super().__init__()
        self.in_features = d_model
        self.out_features = n_tags
        self.hidden = cfg.get("hidden",256)
        self.activation = nn.GELU()
        self.p = cfg.get("dropout",0.0)
        self.linear1 = nn.Linear(self.in_features,self.hidden)
        self.linear2 = nn.Linear(self.hidden,self.out_features)
        self.dropout = nn.Dropout(p=self.p)
        self.crf = CRF(n_tags, batch_first=True)
        
    def forward(self,x:torch.Tensor)->torch.Tensor:
        B,T,C = x.shape
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        logits = self.linear2(x)
        return logits
    def loss(self,logits:torch.Tensor ,tags:torch.Tensor):
        # need to use this logits and find crossentropy error
        mask = tags!=-100
        safe_tags=tags.masked_fill(~mask,0)
        log_likelihood = self.crf(logits,safe_tags,mask=mask,reduction="mean")
        return -log_likelihood
    def decode(self, logits):
        mask = torch.ones(
            logits.shape[:2],
        dtype=torch.bool,
        device=logits.device
    )

        predictions = self.crf.decode(logits, mask=mask)

        return torch.tensor(
        predictions,
        dtype=torch.long,
        device=logits.device
    )






















def build_head(d_model, n_tags, name, **cfg):
    if(name == "linear"):
        return Linear_head(d_model,n_tags,**cfg)
    elif(name ==  "mlp"):
        return Mlp_head(d_model,n_tags,**cfg)
    elif(name == "mlp_crf"):
        return Mlp_Crf__head(d_model,n_tags,**cfg)
    else:
        raise ValueError(f"please provide name of classification head in cfg , availible options : linear , mlp , mlp_crf")
   

In [ ]:
%%writefile /kaggle/working/phase2/techniques/lora.py
"""YOUR FILE: LoRA (Hu et al., 2021) [3].

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); replace_linears(backbone, <which projections>, <your LoRALinear around nn.Linear>)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn
import math
from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,r,alpha):
        super().__init__()
        self.base_layer=base
        self.in_features = base.in_features
        self.out_features = base.out_features
        for param in self.base_layer.parameters():
            param.requires_grad = False
        # define A and B as dimension B is out features cross r and A is r cross in features
        self.A = nn.Parameter(torch.randn(r, self.in_features))
        
        self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features,r))
        self.scaling  = alpha/r
    def forward (self,x):
        base_output = self.base_layer(x)
        scaled_output = (x@self.A.T)@self.B.T
        return base_output+self.scaling*scaled_output
        



def apply(backbone, r, alpha, targets):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: LoRALinear(lin, r, alpha))

In [ ]:
%%writefile /kaggle/working/phase2/techniques/lorma.py
"""YOUR FILE: LoRMA: Low-Rank Multiplicative Adaptation (Bihany, Patel, Modi, 2025) [12] -- COMPULSORY.

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); replace_linears(backbone, <which projections>, <your LoRMALinear around nn.Linear>)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn
import math
from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)

class LoRMALinear(nn.Module):
    def __init__(self,base:nn.Linear,r,alpha):
        super().__init__()
        self.base_layer=base
        self.in_features = base.in_features
        self.out_features = base.out_features
        for param in self.base_layer.parameters():
            param.requires_grad = False
        # define A and B as dimension B is out features cross r and A is r cross in features
        self.A = nn.Parameter(torch.empty(r, self.out_features))
        
        self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features,r))
        self.scaling  = alpha/r
    def forward (self,x):
        base_output = self.base_layer(x)
        scaled_output = (base_output@self.A.T)@self.B.T
        return base_output+self.scaling*scaled_output
        



def apply(backbone, r, alpha, targets):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: LoRMALinear(lin, r, alpha))

In [ ]:
%%writefile /kaggle/working/phase2/techniques/adapters.py
"""YOUR FILE: Houlsby / Pfeiffer bottleneck adapters [6, 7].

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); wrap_modules(backbone, <name endswith '.attn' / '.ffn'>, <your adapter wrapper>)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)


def apply(backbone, **cfg):
    raise NotImplementedError("techniques/adapters.py: implement apply(backbone, **cfg)")

In [ ]:
%%writefile /kaggle/working/phase2/techniques/bitfit.py
"""YOUR FILE: BitFit (Ben Zaken et al., 2022) [8].

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
decide which parameters stay trainable (note: attention projections are bias=False; FFN linears and LayerNorms have biases)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)


def apply(backbone, **cfg):
    freeze(backbone)
    for name , param in backbone.named_parameters():
        if name.endswith("bias") is True:
            param.requires_grad = True
        

In [ ]:
%%writefile /kaggle/working/phase2/techniques/partial_ft.py
"""YOUR FILE: Partial / layer-wise fine-tuning.

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); then unfreeze the parts you choose (backbone.blocks is input -> output order)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)


def apply(backbone,k):
    freeze(backbone)
    if(k>0):
        
    # unfreeze the final norm layer
        if(backbone.kind == "encdec"):
            if backbone.encdec_mode == "encoder":
                unfreeze(backbone.lm.ln_f_enc)
            else:
                unfreeze(backbone.lm.ln_f_dec)


            
           
        else:
            unfreeze(backbone.lm.ln_f)
        for block in  backbone.blocks[-k:]:
            unfreeze(block)
        
        
    
    




   

In [ ]:
%%writefile /kaggle/working/phase2/techniques/kd.py
"""YOUR FILE: knowledge-distillation loss (Hinton et al., 2015) [2] -- COMPULSORY experiment.

Used when the config has a "kd" block: {"teacher_run": <run dir of a finished teacher>, ...your options}.
The plumbing loads the teacher (frozen, eval mode), runs it on exactly the same windows as the student and calls

kd_loss(student_logits, teacher_logits, tags, **cfg) -> scalar
    student_logits, teacher_logits: (B, T, n_tags) float32 (same tag set: checked)
    tags: (B, T) long gold tags, -100 = ignore
    cfg: the "kd" dict minus "teacher_run" (e.g. your temperature / mixing weight)
This loss REPLACES the normal task loss for the student, so include the gold-label term yourself if you want it.
The student's own adaptation is chosen as usual with "technique" (e.g. full_ft on a smaller backbone).
"""

"""YOUR FILE: knowledge-distillation loss (Hinton et al., 2015) [2]."""

import torch
import torch.nn.functional as F


def kd_loss(student_logits, teacher_logits, tags, **cfg):
    """
    Knowledge-distillation loss for token classification / NER.

    Args:
        student_logits: (B, T, C) student logits.
        teacher_logits: (B, T, C) teacher logits.
        tags: (B, T) gold tag IDs; -100 means ignore.
        cfg:
            temperature: softmax temperature (default 2.0).
            alpha: weight for KD loss (default 0.5).
                   Total loss = alpha * KD + (1-alpha) * CE.

    Returns:
        Scalar loss.
    """
    temperature = cfg.get("temperature", 2.0)
    alpha = cfg.get("alpha", 0.5)

    if temperature <= 0:
        raise ValueError("temperature must be positive")

    if not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha must be between 0 and 1")

    if student_logits.shape != teacher_logits.shape:
        raise ValueError(
            "Student and teacher logits must have the same shape"
        )

    mask = tags != -100

  
    if not mask.any():
        return student_logits.sum() * 0.0

    
    teacher_logits = teacher_logits.detach()

  
    student_log_probs = F.log_softmax(
        student_logits / temperature, dim=-1
    )
    teacher_probs = F.softmax(
        teacher_logits / temperature, dim=-1
    )

    
    kl_per_token = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="none"
    ).sum(dim=-1)

    kd_term = kl_per_token[mask].mean() * (temperature ** 2)

   
    ce_term = F.cross_entropy(
        student_logits[mask],
        tags[mask]
    )

   
    return alpha * kd_term + (1.0 - alpha) * ce_term

## Stage 2 — data, Phase-1 inputs, oracle check

In [ ]:
# Phase-2 data from the gated HF dataset (Kaggle secret "HF"), then the Phase-1 inputs
subprocess.run([sys.executable, "fetch_data.py", "--out", os.path.join(WORK, "data")], check=True)
DATA_DIR = os.path.join(WORK, "data", "Phase_2_LNER")

def find_all(pattern):
    return sorted(glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True))

def find_one(pattern):
    hits = find_all(pattern)
    if len(hits) > 1:
        print(f"  note: {len(hits)} matches for {pattern}, using the first: {hits}")
    return hits[0] if hits else None

def sha8(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()[:8]

# every attached tokenizer must be the same file (the models were trained in different notebooks)
TOKS = find_all("legal_bpe.json")
assert TOKS, "legal_bpe.json not found under /kaggle/input (Add Input: your Phase-1 notebook outputs)"
tok_hashes = {t: sha8(t) for t in TOKS}
for t, h in tok_hashes.items():
    print(f"tokenizer {h}  {t}")
assert len(set(tok_hashes.values())) == 1, ("the attached Phase-1 outputs contain DIFFERENT legal_bpe.json files: "
                                            "models trained with different tokenizers cannot share one pipeline")
TOKENIZER = TOKS[0]
print("tokenizer sha256:", tok_hashes[TOKENIZER], "(Phase-1 legal BPE was 38f51f14)")

PHASE1 = {                                     # first match under /kaggle/input; or set explicit paths here
    "A": find_one("run_A/best.pt"),
    "B": find_one("run_B2/best.pt") or find_one("run_B/best.pt"),
    "C": find_one("run_C/best.pt"),
    "D": find_one("run_D/best.pt"),
}
for k, v in PHASE1.items():
    if v is None:
        print(f"Phase-1 {k}: NOT FOUND")
        continue
    ck = torch.load(v, map_location="cpu", weights_only=False)
    c = ck["config"]
    tok_used = c.get("tokenizer_path") or "?"
    print(f"Phase-1 {k}: {v}\n    tag={c.get('tag')} module={c['module']} step={ck.get('step')} "
          f"best_val={ck.get('best_val')} | trained with tokenizer at {tok_used}")
    del ck
missing = [k for k, v in PHASE1.items() if v is None]
if missing:
    print("WARNING: no checkpoint found for", missing, "-> runs using them will fail (Add Input: that notebook's output)")

In [ ]:
# oracle check (no model, ~1-2 min): gold tags through the real windows/merge/decode/offset/JSON path.
# Expected: macro-F1 0.9945 with split_punct=True, 0.9866 with False (the char<->token alignment ceiling).
subprocess.run([sys.executable, "check_pipeline.py", "--data", DATA_DIR, "--tokenizer", TOKENIZER], check=True)

## Stage 3 — configuration (yours)

In [ ]:
# =====================================================================================
# YOUR CHOICES (course rule 9). Values below are yours (transcribed from your config).
# =====================================================================================
import copy

BASE = {
    "data_dir": DATA_DIR,
    "tokenizer": TOKENIZER,
    "out_dir": os.path.join(WORK, "runs"),
    "fast_attention": True,
    "backbone": {"prefix": 640, "encdec_mode": None, "truncate_layers": None, "dropout": 0.1},
    "preprocess": {"split_punct": True, "tag_scheme": "BIO"},
    "split": {"mode": "random", "val_frac": 0.15, "seed": 42},
    # default windows for A, B, D (C overrides them below: encoder limit 640)
    "window": {"train_len": 768, "train_stride": 512, "infer_len": 768, "infer_stride": 512, "merge": "center"},
    "head": {"name": "linear", "dropout": 0.1},
    "technique": {"name": "full_ft"},
    "kd": None,
    "train": {
        "epochs": 8, "batch_size": 16, "micro_batch": 2,
        "lr": 2e-5, "head_lr": 1e-4,                      # full fine-tuning baseline
        "weight_decay": 0.01, "warmup_frac": 0.06, "schedule": "cosine", "grad_clip": 1.0,
        "class_weights": "inv_sqrt", "o_weight": None,
        "patience": 2, "max_minutes": 80, "seed": 42,
        "eval_every_epochs": 1, "log_every": 20, "infer_micro_batch": 2,
    },
    "decode": {"repair": "strict"},
}

LORA_TARGETS = ["q_proj", "v_proj"]       # these names exist in A, B, C and D (checked)
C_WINDOW = {"train_len": 512, "train_stride": 384, "infer_len": 512, "infer_stride": 384, "merge": "center"}

EXPERIMENTS = {
    # 1. B: full fine-tuning baseline
    "B_full":   {"model": "B", "technique": {"name": "full_ft"}},
    # 2. B: LoRMA
    "B_lorma":  {"model": "B", "technique": {"name": "lorma", "r": 8, "alpha": 16, "targets": LORA_TARGETS},
                 "train": {"lr": 2e-4}},
    # 3. C (encoder mode): full fine-tuning
    "C_full":   {"model": "C", "backbone": {"encdec_mode": "encoder"}, "window": C_WINDOW,
                 "technique": {"name": "full_ft"}},
    # 4. C (encoder mode): LoRMA
    "C_lorma":  {"model": "C", "backbone": {"encdec_mode": "encoder"}, "window": C_WINDOW,
                 "technique": {"name": "lorma", "r": 8, "alpha": 16, "targets": LORA_TARGETS},
                 "train": {"lr": 2e-4}},
    # 5. A: LoRA
    "A_lora":   {"model": "A", "technique": {"name": "lora", "r": 8, "alpha": 16, "targets": LORA_TARGETS},
                 "train": {"lr": 1e-4}},
    # 6. B: partial fine-tuning (top 2 blocks + final norm) with the CRF head
    "B_peft":   {"model": "B", "technique": {"name": "partial_ft", "k": 2},
                 "head": {"name": "mlp_crf", "hidden": 256, "dropout": 0.1}},
    # 7. D (prefix-LM mask over the first 640 tokens): BitFit (the other one of partial_ft / bitfit)
    "D_peft":   {"model": "D", "backbone": {"prefix": 640}, "technique": {"name": "bitfit"},
                 "train": {"lr": 5e-5}},
    # --- knowledge distillation: disabled for now. To enable later: uncomment, fill the Nones, add "KD" to RUN.
    # "KD": {"model": None, "technique": {"name": None}, "backbone": {"truncate_layers": None},
    #        "kd": {"teacher_run": os.path.join(WORK, "runs", "<teacher name>"), "temperature": None, "alpha": None}},
}

# queue order: the two GPUs take jobs from the front (long full-FT runs first)
RUN = ["B_full", "C_full", "B_lorma", "C_lorma", "A_lora", "B_peft", "D_peft"]

def deep_update(d, u):
    for k, v in u.items():
        if isinstance(v, dict) and isinstance(d.get(k), dict):
            deep_update(d[k], v)
        else:
            d[k] = copy.deepcopy(v)
    return d

import train_ner, backbones
CFG_PATHS = []
for name in RUN:
    exp = copy.deepcopy(EXPERIMENTS[name])
    model = exp.pop("model")
    assert model in PHASE1, f"{name}: set 'model' to one of {list(PHASE1)}"
    assert exp["technique"].get("name") is not None, f"{name}: set technique name"
    assert exp["technique"]["name"] != "partial_ft" or exp["technique"].get("k") is not None,         f"{name}: partial_ft needs 'k'"
    cfg = deep_update(copy.deepcopy(BASE), exp)
    cfg["name"] = name
    cfg["phase1_ckpt"] = PHASE1[model]
    assert cfg["phase1_ckpt"], f"{name}: no Phase-1 checkpoint found for model {model}"
    train_ner.check_config(cfg)              # required keys + technique options vs YOUR apply() signature
    # window vs the backbone's real max length (read from the checkpoint config, no weights loaded)
    _, kind, spec, _ = backbones.load_phase1(cfg["phase1_ckpt"], load_weights=False)
    if kind == "encdec":
        limit = spec["encoder"]["max_seq_len"] if cfg["backbone"]["encdec_mode"] == "encoder" else             min(spec["encoder"]["max_seq_len"], spec["decoder"]["max_seq_len"])
    else:
        limit = spec["model"]["max_seq_len"]
    w = cfg["window"]
    assert max(w["train_len"], w["infer_len"]) <= limit, f"{name}: window > {limit} (max length of model {model})"
    path = os.path.join(WORK, "configs", f"{name}.json")
    json.dump(cfg, open(path, "w"), indent=2)
    CFG_PATHS.append(path)
    print(f"{name:10s} | model={model} ({kind}) | technique={cfg['technique']['name']:10s} | "
          f"head={cfg['head']['name']:8s} | window={w['train_len']}/{limit} | lr={cfg['train']['lr']:.1e}")
_techs = {json.load(open(p))["technique"]["name"] for p in CFG_PATHS}
print(f"{len(CFG_PATHS)} configs validated and written | distinct techniques: {sorted(_techs)} ({len(_techs)})")
if len(_techs) < 5:
    print("WARNING: fewer than 5 distinct fine-tuning techniques in RUN (Phase 2 requires >= 5 + KD)")

## Stage 4 — runs (both GPUs)

In [ ]:
# Runs every config in RUN on BOTH T4s in parallel: one training process per GPU at a time, two fine-tunings at once.
# Shared queue: whichever GPU frees up first takes the next runnable config (in RUN order). A KD run is skipped until its
# teacher has finished (summary.json), so it never blocks a GPU. Status every 5 min; logs: runs/<name>.stdout,
# runs/<name>/log.jsonl. Running `nvidia-smi` in another cell while this runs should show one python process per GPU.
assert N_GPUS >= 1, "no GPU visible: set Accelerator to 'GPU T4 x2'"
if N_GPUS < 2:
    print("WARNING: only", N_GPUS, "GPU visible -> runs will be sequential. Set Accelerator to 'GPU T4 x2'.")
gpus = ",".join(str(i) for i in range(N_GPUS))
print("launching on GPUs", gpus)
proc = subprocess.run([sys.executable, "run_many.py", "--gpus", gpus, "--every", "300", *CFG_PATHS])
print(f"run_many exit code {proc.returncode} (non-zero = at least one run failed: see the summary above)")
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h")

## Stage 5 — results and zips

In [ ]:
# results table + all zips in /kaggle/working/submissions
import pandas as pd
rows = []
os.makedirs("/kaggle/working/submissions", exist_ok=True)
for sp in sorted(glob.glob(os.path.join(WORK, "runs", "*", "summary.json"))):
    s = json.load(open(sp))
    rows.append(dict(name=s["name"], phase1=s["phase1"].get("tag") or s["phase1"]["module"],
                     technique=s["technique"]["name"], head=s["head"]["name"], kd=s["kd"],
                     val_macro_f1=s.get("val_macro_f1"), val_macro_p=s.get("val_macro_p"), val_macro_r=s.get("val_macro_r"),
                     trainable=s["params"]["trainable"], trainable_pct=s["params"]["trainable_pct"],
                     total_params=s["params"]["total"], best_epoch=s["best_epoch"],
                     train_min=round(s["train_seconds"] / 60, 1), peak_GB=s["peak_mem_GB"],
                     infer_tok_s=s["test_infer_tok_s"], test_spans=s["test_spans"]))
    z = os.path.join(os.path.dirname(sp), f"submission_{s['name']}.zip")
    if os.path.exists(z):
        shutil.copy(z, "/kaggle/working/submissions/")
df = pd.DataFrame(rows)
display(df)
df.to_csv("/kaggle/working/submissions/summary.csv", index=False)
print(sorted(os.listdir("/kaggle/working/submissions")))

In [ ]:
# per-label strict scores on the held-out docs (counts only; interpretation is yours)
for vp in sorted(glob.glob(os.path.join(WORK, "runs", "*", "val_scores.json"))):
    v = json.load(open(vp))
    print(os.path.basename(os.path.dirname(vp)), "macro-F1", v["macro_f1"])
    display(pd.DataFrame(v["per_label"]).T)